<a href="https://colab.research.google.com/github/kaiser62/colab_files/blob/main/telebotuploadmasa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install Dependencies
!pip install pyffmpeg python-telegram-bot tqdm cryptg telethon imageio Pillow  pypdl


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.1/676.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.3/238.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.9/715.9 kB 37.5 MB/s eta 0:00:00
  Created wheel for pyaes: filename=pyaes-1.6.1-py3-none-any.whl size=26346 sha256=117c3d4ed977c114e2c82a9d88c2b4bd37da44823888d884368e287351882040
  Stored in directory: /root/.cache/pip/wheels/4e/52/33/010d0843550bffb6a591b11629070ae140c0ad4f53e68a3bd3
Successfully built pyaes


# OLD Section

In [ ]:
import logging
import asyncio
import requests
import os
import subprocess
from telegram import Bot, InputFile
import nest_asyncio

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

bot_token = '1559476692:AAGKce57g7WKPU_sKcOUOhGQP6bY18Ogu9o'
channel_id = -1002296651104
bot = Bot(token=bot_token)

# Define URL list file (update as needed)
url_list = '/content/url.txt'  # Adjust path based on Colab

def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []

def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None

def download_video(url, download_folder='/content/downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None

def make_streamable(input_file, output_folder='/content/streamable'):
    try:
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, os.path.basename(input_file))
        command = [
            'ffmpeg', '-y', '-i', input_file,
            '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '28',
            '-c:a', 'aac', '-b:a', '128k',
            '-movflags', '+faststart', '-vf', 'format=yuv420p', output_file
        ]
        subprocess.run(command, check=True)
        return output_file
    except Exception as e:
        logger.error(f"Error converting video {input_file} to streamable format: {e}")
        return None

def split_video(input_file, output_folder='/content/split_videos', part_size_mb=40):
    try:
        os.makedirs(output_folder, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        output_pattern = os.path.join(output_folder, f"{base_name}_part_%03d.mp4")

        # Get video duration in seconds
        probe = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                                '-show_entries', 'format=duration', '-of', 'csv=p=0', input_file],
                               capture_output=True, text=True)
        duration = float(probe.stdout.strip())

        # Get input file size in MB
        input_size_mb = os.path.getsize(input_file) / (1024 * 1024)

        # Calculate approximate duration per 40MB chunk
        chunk_duration = (duration / input_size_mb) * part_size_mb

        # FFmpeg command to split video into ~40MB chunks
        command = [
            'ffmpeg', '-y', '-i', input_file, '-c', 'copy', '-map', '0',
            '-f', 'segment', '-segment_time', str(int(chunk_duration)),
            '-reset_timestamps', '1', output_pattern
        ]

        result = subprocess.run(command, capture_output=True, text=True, check=False)

        if result.returncode != 0:
            logger.error(f"FFmpeg error: {result.stderr}")
            return []

        return [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.startswith(base_name)]

    except Exception as e:
        logger.error(f"Error splitting video {input_file}: {e}")
        return []

async def send_video(file_path):
    try:
        with open(file_path, 'rb') as video_file:
            await bot.send_video(chat_id=channel_id, video=InputFile(video_file, filename=os.path.basename(file_path)),
                                 supports_streaming=True)
        os.remove(file_path)  # Delete file after sending
        logger.info(f"Deleted {file_path} after upload")
    except Exception as e:
        logger.error(f"Failed to send video from {file_path}: {e}")


async def process_and_send_videos():
    video_urls = read_video_urls()
    for url in video_urls:
        file_size = check_file_size(url)
        if file_size and file_size > 200:
            continue
        video_file = download_video(url)
        if video_file:
            streamable_file = make_streamable(video_file)
            os.remove(video_file)  # Delete the original downloaded video
            logger.info(f"Deleted {video_file} after processing")

            if streamable_file:
                if os.path.getsize(streamable_file) > 40 * 1024 * 1024:
                    parts = split_video(streamable_file)
                    os.remove(streamable_file)  # Delete the full processed video after splitting
                    logger.info(f"Deleted {streamable_file} after splitting")

                    for part in parts:
                        await send_video(part)
                else:
                    await send_video(streamable_file)
                    #os.remove(streamable_file)  # Delete the processed video if not split
                    logger.info(f"Deleted {streamable_file} after upload")


if __name__ == '__main__':
    nest_asyncio.apply() # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())


In [ ]:
# @title bot token upload
import logging
import asyncio
import requests
from telegram import Bot
from telegram import InputFile
import os
import subprocess
import nest_asyncio

# Enable logging to help with debugging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Replace with your bot's token
bot_token = '1559476692:AAGKce57g7WKPU_sKcOUOhGQP6bY18Ogu9o'

# Replace with your channel ID
channel_id = -1002296651104  # This is the numeric ID of your channel

# Initialize the bot
bot = Bot(token=bot_token)

# Declare the URL list file path
url_list = '/content/a.txt' # Change this path as needed


# Function to read URLs from the specified file
def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        logger.info(f"Read {len(urls)} URLs from {filename}")
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


# Function to download the video
def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])  # Save video with its name
        response = requests.get(url, stream=True)

        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
            logger.info(f"Downloaded video: {file_name}")
            return file_name
        else:
            logger.error(f"Failed to download video from {url}")
            return None
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
        return None


# Function to convert video to a streamable format (MP4 with H.264 codec)
def convert_to_streamable(input_file, output_folder='converted'):
    try:
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, os.path.basename(input_file).split('.')[0] + '_streamable.mp4')

        # Use ffmpeg to convert the video to a streamable format
        command = [
            'ffmpeg', '-i', input_file,
            '-c:v', 'libx264', '-c:a', 'aac',  # Use H.264 codec for video and AAC for audio
            '-strict', 'experimental',  # Allow experimental AAC codec
            '-movflags', '+faststart',  # Enable streaming by moving metadata to the beginning
            output_file
        ]

        subprocess.run(command, check=True)
        logger.info(f"Converted video to streamable format: {output_file}")
        return output_file
    except Exception as e:
        logger.error(f"Error converting video {input_file} to streamable format: {e}")
        return None


# Async function to send video to the channel
async def send_video_from_file(file_path):
    try:
        with open(file_path, 'rb') as video_file:
            await bot.send_video(chat_id=channel_id, video=InputFile(video_file, filename=os.path.basename(file_path)),
                                 supports_streaming=True)
        logger.info(f"Successfully sent video from {file_path} to channel ID {channel_id}")
    except Exception as e:
        logger.error(f"Failed to send video from {file_path}: {e}")


# Main function to download, convert, and send videos
async def download_and_send_videos():
    video_urls = read_video_urls(url_list)  # Read the URLs from the text file

    if not video_urls:
        logger.error("No URLs to process.")
        return

    for url in video_urls:
        logger.info(f"Processing video from URL: {url}")

        # Download the video
        downloaded_video = download_video(url)

        if downloaded_video:
            # Convert the video to a streamable format
            streamable_video = convert_to_streamable(downloaded_video)

            if streamable_video:
                # Send the converted video
                await send_video_from_file(streamable_video)
                os.remove(downloaded_video)  # Delete the original downloaded video
                os.remove(streamable_video)  # Delete the converted video after sending
            else:
                logger.error(f"Skipping video due to conversion failure: {url}")
        else:
            logger.error(f"Skipping video due to download failure: {url}")


# Run the async function
if __name__ == '__main__':
    nest_asyncio.apply() # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(download_and_send_videos())

ERROR:__main__:Error converting video downloads/58617.mp4 to streamable format: Command '['ffmpeg', '-i', 'downloads/58617.mp4', '-c:v', 'libx264', '-c:a', 'aac', '-strict', 'experimental', '-movflags', '+faststart', 'converted/58617_streamable.mp4']' returned non-zero exit status 1.
ERROR:__main__:Skipping video due to conversion failure: https://server10.masahub.cc/myfiless/id/58617.mp4
ERROR:__main__:Error converting video downloads/58594.mp4 to streamable format: Command '['ffmpeg', '-i', 'downloads/58594.mp4', '-c:v', 'libx264', '-c:a', 'aac', '-strict', 'experimental', '-movflags', '+faststart', 'converted/58594_streamable.mp4']' returned non-zero exit status 1.
ERROR:__main__:Skipping video due to conversion failure: https://server10.masahub.cc/myfiless/id/58594.mp4
ERROR:__main__:Error converting video downloads/58600.mp4 to streamable format: Command '['ffmpeg', '-i', 'downloads/58600.mp4', '-c:v', 'libx264', '-c:a', 'aac', '-strict', 'experimental', '-movflags', '+faststart',

# 2. FastTelethon.py

In [ ]:
# @title modified
%%writefile /content/FastTelethon.py
# copied from https://github.com/tulir/mautrix-telegram/blob/master/mautrix_telegram/util/parallel_file_transfer.py
# Copyright (C) 2021 Tulir Asokan
import asyncio
import hashlib
import inspect
import logging
import math
import os
from collections import defaultdict
from typing import Optional, List, AsyncGenerator, Union, Awaitable, DefaultDict, Tuple, BinaryIO

from telethon import utils, helpers, TelegramClient
from telethon.crypto import AuthKey
from telethon.network import MTProtoSender
from telethon.tl.alltlobjects import LAYER
from telethon.tl.functions import InvokeWithLayerRequest
from telethon.tl.functions.auth import ExportAuthorizationRequest, ImportAuthorizationRequest
from telethon.tl.functions.upload import (GetFileRequest, SaveFilePartRequest,
                                          SaveBigFilePartRequest)
from telethon.tl.types import (Document, InputFileLocation, InputDocumentFileLocation,
                               InputPhotoFileLocation, InputPeerPhotoFileLocation, TypeInputFile,
                               InputFileBig, InputFile)

try:
    from mautrix.crypto.attachments import async_encrypt_attachment
except ImportError:
    async_encrypt_attachment = None

log: logging.Logger = logging.getLogger("telethon")

TypeLocation = Union[Document, InputDocumentFileLocation, InputPeerPhotoFileLocation,
                     InputFileLocation, InputPhotoFileLocation]


class DownloadSender:
    client: TelegramClient
    sender: MTProtoSender
    request: GetFileRequest
    remaining: int
    stride: int

    def __init__(self, client: TelegramClient, sender: MTProtoSender, file: TypeLocation, offset: int, limit: int,
                 stride: int, count: int) -> None:
        self.sender = sender
        self.client = client
        self.request = GetFileRequest(file, offset=offset, limit=limit)
        self.stride = stride
        self.remaining = count

    async def next(self) -> Optional[bytes]:
        if not self.remaining:
            return None
        result = await self.client._call(self.sender, self.request)
        self.remaining -= 1
        self.request.offset += self.stride
        return result.bytes

    def disconnect(self) -> Awaitable[None]:
        return self.sender.disconnect()


class UploadSender:
    client: TelegramClient
    sender: MTProtoSender
    request: Union[SaveFilePartRequest, SaveBigFilePartRequest]
    part_count: int
    stride: int
    previous: Optional[asyncio.Task]
    loop: asyncio.AbstractEventLoop

    def __init__(self, client: TelegramClient, sender: MTProtoSender, file_id: int, part_count: int, big: bool,
                 index: int,
                 stride: int, loop: asyncio.AbstractEventLoop) -> None:
        self.client = client
        self.sender = sender
        self.part_count = part_count
        if big:
            self.request = SaveBigFilePartRequest(file_id, index, part_count, b"")
        else:
            self.request = SaveFilePartRequest(file_id, index, b"")
        self.stride = stride
        self.previous = None
        self.loop = loop

    async def next(self, data: bytes) -> None:
        if self.previous:
            await self.previous
        self.previous = self.loop.create_task(self._next(data))

    async def _next(self, data: bytes) -> None:
        self.request.bytes = data
        log.debug(f"Sending file part {self.request.file_part}/{self.part_count}"
                  f" with {len(data)} bytes")
        await self.client._call(self.sender, self.request)
        self.request.file_part += self.stride

    async def disconnect(self) -> None:
        if self.previous:
            await self.previous
        return await self.sender.disconnect()


class ParallelTransferrer:
    client: TelegramClient
    loop: asyncio.AbstractEventLoop
    dc_id: int
    senders: Optional[List[Union[DownloadSender, UploadSender]]]
    auth_key: AuthKey
    upload_ticker: int

    def __init__(self, client: TelegramClient, dc_id: Optional[int] = None) -> None:
        self.client = client
        self.loop = self.client.loop
        self.dc_id = dc_id or self.client.session.dc_id
        self.auth_key = (None if dc_id and self.client.session.dc_id != dc_id
                         else self.client.session.auth_key)
        self.senders = None
        self.upload_ticker = 0

    async def _cleanup(self) -> None:
        await asyncio.gather(*[sender.disconnect() for sender in self.senders])
        self.senders = None

    @staticmethod
    def _get_connection_count(file_size: int, max_count: int = 30,  # Increased for large files
                              full_size: int = 300 * 1024 * 1024) -> int:
        if file_size <= full_size:
            # Smaller files: logarithmic scaling
            return min(30, max(4, int(math.log(file_size + 1, 2) * 6)))
        else:
            # Large files: linear scaling with a higher max
            # Ensure at least 30 workers for large files, up to 50
            additional_workers = min(20, int((file_size - full_size) / (100 * 1024 * 1024)))
            return 30 + additional_workers

    async def _init_download(self, connections: int, file: TypeLocation, part_count: int,
                             part_size: int) -> None:
        minimum, remainder = divmod(part_count, connections)

        def get_part_count() -> int:
            nonlocal remainder
            if remainder > 0:
                remainder -= 1
                return minimum + 1
            return minimum

        # The first cross-DC sender will export+import the authorization, so we always create it
        # before creating any other senders.
        self.senders = [
            await self._create_download_sender(file, 0, part_size, connections * part_size,
                                               get_part_count()),
            *await asyncio.gather(
                *[self._create_download_sender(file, i, part_size, connections * part_size,
                                               get_part_count())
                  for i in range(1, connections)])
        ]

    async def _create_download_sender(self, file: TypeLocation, index: int, part_size: int,
                                      stride: int,
                                      part_count: int) -> DownloadSender:
        return DownloadSender(self.client, await self._create_sender(), file, index * part_size, part_size,
                              stride, part_count)

    async def _init_upload(self, connections: int, file_id: int, part_count: int, big: bool
                           ) -> None:
        self.senders = [
            await self._create_upload_sender(file_id, part_count, big, 0, connections),
            *await asyncio.gather(
                *[self._create_upload_sender(file_id, part_count, big, i, connections)
                  for i in range(1, connections)])
        ]

    async def _create_upload_sender(self, file_id: int, part_count: int, big: bool, index: int,
                                    stride: int) -> UploadSender:
        return UploadSender(self.client, await self._create_sender(), file_id, part_count, big, index, stride,
                            loop=self.loop)

    async def _create_sender(self) -> MTProtoSender:
        dc = await self.client._get_dc(self.dc_id)
        sender = MTProtoSender(self.auth_key, loggers=self.client._log)
        await sender.connect(self.client._connection(dc.ip_address, dc.port, dc.id,
                                                     loggers=self.client._log,
                                                     proxy=self.client._proxy))
        if not self.auth_key:
            log.debug(f"Exporting auth to DC {self.dc_id}")
            auth = await self.client(ExportAuthorizationRequest(self.dc_id))
            self.client._init_request.query = ImportAuthorizationRequest(id=auth.id,
                                                                         bytes=auth.bytes)
            req = InvokeWithLayerRequest(LAYER, self.client._init_request)
            await sender.send(req)
            self.auth_key = sender.auth_key
        return sender

    async def init_upload(self, file_id: int, file_size: int, part_size_kb: Optional[float] = None,
                          connection_count: Optional[int] = None) -> Tuple[int, int, bool]:
        connection_count = connection_count or self._get_connection_count(file_size)
        part_size = (part_size_kb or utils.get_appropriated_part_size(file_size)) * 1024
        part_count = (file_size + part_size - 1) // part_size
        is_large = file_size > 10 * 1024 * 1024
        await self._init_upload(connection_count, file_id, part_count, is_large)
        return part_size, part_count, is_large

    async def upload(self, part: bytes) -> None:
        await self.senders[self.upload_ticker].next(part)
        self.upload_ticker = (self.upload_ticker + 1) % len(self.senders)

    async def finish_upload(self) -> None:
        await self._cleanup()

    async def download(self, file: TypeLocation, file_size: int,
                       part_size_kb: Optional[float] = None,
                       connection_count: Optional[int] = None) -> AsyncGenerator[bytes, None]:
        connection_count = connection_count or self._get_connection_count(file_size)
        part_size = (part_size_kb or utils.get_appropriated_part_size(file_size)) * 1024
        part_count = math.ceil(file_size / part_size)
        log.debug("Starting parallel download: "
                  f"{connection_count} {part_size} {part_count} {file!s}")
        await self._init_download(connection_count, file, part_count, part_size)

        part = 0
        while part < part_count:
            tasks = []
            for sender in self.senders:
                tasks.append(self.loop.create_task(sender.next()))
            for task in tasks:
                data = await task
                if not data:
                    break
                yield data
                part += 1
                log.debug(f"Part {part} downloaded")

        log.debug("Parallel download finished, cleaning up connections")
        await self._cleanup()


parallel_transfer_locks: DefaultDict[int, asyncio.Lock] = defaultdict(lambda: asyncio.Lock())


def stream_file(file_to_stream: BinaryIO, chunk_size=1024):
    while True:
        data_read = file_to_stream.read(chunk_size)
        if not data_read:
            break
        yield data_read


async def _internal_transfer_to_telegram(client: TelegramClient,
                                         response: BinaryIO,
                                         progress_callback: callable
                                         ) -> Tuple[TypeInputFile, int]:
    file_id = helpers.generate_random_long()
    file_size = os.path.getsize(response.name)

    hash_md5 = hashlib.md5()
    uploader = ParallelTransferrer(client)
    part_size, part_count, is_large = await uploader.init_upload(file_id, file_size)
    buffer = bytearray()
    for data in stream_file(response):
        if progress_callback:
            r = progress_callback(response.tell(), file_size)
            if inspect.isawaitable(r):
                await r
        if not is_large:
            hash_md5.update(data)
        if len(buffer) == 0 and len(data) == part_size:
            await uploader.upload(data)
            continue
        new_len = len(buffer) + len(data)
        if new_len >= part_size:
            cutoff = part_size - len(buffer)
            buffer.extend(data[:cutoff])
            await uploader.upload(bytes(buffer))
            buffer.clear()
            buffer.extend(data[cutoff:])
        else:
            buffer.extend(data)
    if len(buffer) > 0:
        await uploader.upload(bytes(buffer))
    await uploader.finish_upload()
    if is_large:
        return InputFileBig(file_id, part_count, "upload"), file_size
    else:
        return InputFile(file_id, part_count, "upload", hash_md5.hexdigest()), file_size


async def download_file(client: TelegramClient,
                        location: TypeLocation,
                        out: BinaryIO,
                        progress_callback: callable = None
                        ) -> BinaryIO:
    size = location.size
    dc_id, location = utils.get_input_location(location)
    # We lock the transfers because telegram has connection count limits
    downloader = ParallelTransferrer(client, dc_id)
    downloaded = downloader.download(location, size)
    async for x in downloaded:
        out.write(x)
        if progress_callback:
            r = progress_callback(out.tell(), size)
            if inspect.isawaitable(r):
                await r

    return out


async def upload_file(client: TelegramClient,
                      file: BinaryIO,
                      progress_callback: callable = None,

                      ) -> TypeInputFile:
    res = (await _internal_transfer_to_telegram(client, file, progress_callback))[0]
    return res


Writing /content/FastTelethon.py


In [ ]:
# @title Original
%%writefile /content/FastTelethon.py
# copied from https://github.com/tulir/mautrix-telegram/blob/master/mautrix_telegram/util/parallel_file_transfer.py
# Copyright (C) 2021 Tulir Asokan
import asyncio
import hashlib
import inspect
import logging
import math
import os
from collections import defaultdict
from typing import Optional, List, AsyncGenerator, Union, Awaitable, DefaultDict, Tuple, BinaryIO

from telethon import utils, helpers, TelegramClient
from telethon.crypto import AuthKey
from telethon.network import MTProtoSender
from telethon.tl.alltlobjects import LAYER
from telethon.tl.functions import InvokeWithLayerRequest
from telethon.tl.functions.auth import ExportAuthorizationRequest, ImportAuthorizationRequest
from telethon.tl.functions.upload import (GetFileRequest, SaveFilePartRequest,
                                          SaveBigFilePartRequest)
from telethon.tl.types import (Document, InputFileLocation, InputDocumentFileLocation,
                               InputPhotoFileLocation, InputPeerPhotoFileLocation, TypeInputFile,
                               InputFileBig, InputFile)

try:
    from mautrix.crypto.attachments import async_encrypt_attachment
except ImportError:
    async_encrypt_attachment = None

log: logging.Logger = logging.getLogger("telethon")

TypeLocation = Union[Document, InputDocumentFileLocation, InputPeerPhotoFileLocation,
                     InputFileLocation, InputPhotoFileLocation]


class DownloadSender:
    client: TelegramClient
    sender: MTProtoSender
    request: GetFileRequest
    remaining: int
    stride: int

    def __init__(self, client: TelegramClient, sender: MTProtoSender, file: TypeLocation, offset: int, limit: int,
                 stride: int, count: int) -> None:
        self.sender = sender
        self.client = client
        self.request = GetFileRequest(file, offset=offset, limit=limit)
        self.stride = stride
        self.remaining = count

    async def next(self) -> Optional[bytes]:
        if not self.remaining:
            return None
        result = await self.client._call(self.sender, self.request)
        self.remaining -= 1
        self.request.offset += self.stride
        return result.bytes

    def disconnect(self) -> Awaitable[None]:
        return self.sender.disconnect()


class UploadSender:
    client: TelegramClient
    sender: MTProtoSender
    request: Union[SaveFilePartRequest, SaveBigFilePartRequest]
    part_count: int
    stride: int
    previous: Optional[asyncio.Task]
    loop: asyncio.AbstractEventLoop

    def __init__(self, client: TelegramClient, sender: MTProtoSender, file_id: int, part_count: int, big: bool,
                 index: int,
                 stride: int, loop: asyncio.AbstractEventLoop) -> None:
        self.client = client
        self.sender = sender
        self.part_count = part_count
        if big:
            self.request = SaveBigFilePartRequest(file_id, index, part_count, b"")
        else:
            self.request = SaveFilePartRequest(file_id, index, b"")
        self.stride = stride
        self.previous = None
        self.loop = loop

    async def next(self, data: bytes) -> None:
        if self.previous:
            await self.previous
        self.previous = self.loop.create_task(self._next(data))

    async def _next(self, data: bytes) -> None:
        self.request.bytes = data
        log.debug(f"Sending file part {self.request.file_part}/{self.part_count}"
                  f" with {len(data)} bytes")
        await self.client._call(self.sender, self.request)
        self.request.file_part += self.stride

    async def disconnect(self) -> None:
        if self.previous:
            await self.previous
        return await self.sender.disconnect()


class ParallelTransferrer:
    client: TelegramClient
    loop: asyncio.AbstractEventLoop
    dc_id: int
    senders: Optional[List[Union[DownloadSender, UploadSender]]]
    auth_key: AuthKey
    upload_ticker: int

    def __init__(self, client: TelegramClient, dc_id: Optional[int] = None) -> None:
        self.client = client
        self.loop = self.client.loop
        self.dc_id = dc_id or self.client.session.dc_id
        self.auth_key = (None if dc_id and self.client.session.dc_id != dc_id
                         else self.client.session.auth_key)
        self.senders = None
        self.upload_ticker = 0

    async def _cleanup(self) -> None:
        await asyncio.gather(*[sender.disconnect() for sender in self.senders])
        self.senders = None

    @staticmethod
    def _get_connection_count(file_size: int, max_count: int = 10,
                              full_size: int = 100 * 1024 * 1024) -> int:
        if file_size > full_size:
            return max_count
        return math.ceil((file_size / full_size) * max_count)

    async def _init_download(self, connections: int, file: TypeLocation, part_count: int,
                             part_size: int) -> None:
        minimum, remainder = divmod(part_count, connections)

        def get_part_count() -> int:
            nonlocal remainder
            if remainder > 0:
                remainder -= 1
                return minimum + 1
            return minimum

        # The first cross-DC sender will export+import the authorization, so we always create it
        # before creating any other senders.
        self.senders = [
            await self._create_download_sender(file, 0, part_size, connections * part_size,
                                               get_part_count()),
            *await asyncio.gather(
                *[self._create_download_sender(file, i, part_size, connections * part_size,
                                               get_part_count())
                  for i in range(1, connections)])
        ]

    async def _create_download_sender(self, file: TypeLocation, index: int, part_size: int,
                                      stride: int,
                                      part_count: int) -> DownloadSender:
        return DownloadSender(self.client, await self._create_sender(), file, index * part_size, part_size,
                              stride, part_count)

    async def _init_upload(self, connections: int, file_id: int, part_count: int, big: bool
                           ) -> None:
        self.senders = [
            await self._create_upload_sender(file_id, part_count, big, 0, connections),
            *await asyncio.gather(
                *[self._create_upload_sender(file_id, part_count, big, i, connections)
                  for i in range(1, connections)])
        ]

    async def _create_upload_sender(self, file_id: int, part_count: int, big: bool, index: int,
                                    stride: int) -> UploadSender:
        return UploadSender(self.client, await self._create_sender(), file_id, part_count, big, index, stride,
                            loop=self.loop)

    async def _create_sender(self) -> MTProtoSender:
        dc = await self.client._get_dc(self.dc_id)
        sender = MTProtoSender(self.auth_key, loggers=self.client._log)
        await sender.connect(self.client._connection(dc.ip_address, dc.port, dc.id,
                                                     loggers=self.client._log,
                                                     proxy=self.client._proxy))
        if not self.auth_key:
            log.debug(f"Exporting auth to DC {self.dc_id}")
            auth = await self.client(ExportAuthorizationRequest(self.dc_id))
            self.client._init_request.query = ImportAuthorizationRequest(id=auth.id,
                                                                         bytes=auth.bytes)
            req = InvokeWithLayerRequest(LAYER, self.client._init_request)
            await sender.send(req)
            self.auth_key = sender.auth_key
        return sender

    async def init_upload(self, file_id: int, file_size: int, part_size_kb: Optional[float] = None,
                          connection_count: Optional[int] = None) -> Tuple[int, int, bool]:
        connection_count = connection_count or self._get_connection_count(file_size)
        part_size = (part_size_kb or utils.get_appropriated_part_size(file_size)) * 1024
        part_count = (file_size + part_size - 1) // part_size
        is_large = file_size > 10 * 1024 * 1024
        await self._init_upload(connection_count, file_id, part_count, is_large)
        return part_size, part_count, is_large

    async def upload(self, part: bytes) -> None:
        await self.senders[self.upload_ticker].next(part)
        self.upload_ticker = (self.upload_ticker + 1) % len(self.senders)

    async def finish_upload(self) -> None:
        await self._cleanup()

    async def download(self, file: TypeLocation, file_size: int,
                       part_size_kb: Optional[float] = None,
                       connection_count: Optional[int] = None) -> AsyncGenerator[bytes, None]:
        connection_count = connection_count or self._get_connection_count(file_size)
        part_size = (part_size_kb or utils.get_appropriated_part_size(file_size)) * 1024
        part_count = math.ceil(file_size / part_size)
        log.debug("Starting parallel download: "
                  f"{connection_count} {part_size} {part_count} {file!s}")
        await self._init_download(connection_count, file, part_count, part_size)

        part = 0
        while part < part_count:
            tasks = []
            for sender in self.senders:
                tasks.append(self.loop.create_task(sender.next()))
            for task in tasks:
                data = await task
                if not data:
                    break
                yield data
                part += 1
                log.debug(f"Part {part} downloaded")

        log.debug("Parallel download finished, cleaning up connections")
        await self._cleanup()


parallel_transfer_locks: DefaultDict[int, asyncio.Lock] = defaultdict(lambda: asyncio.Lock())


def stream_file(file_to_stream: BinaryIO, chunk_size=1024):
    while True:
        data_read = file_to_stream.read(chunk_size)
        if not data_read:
            break
        yield data_read


async def _internal_transfer_to_telegram(client: TelegramClient,
                                         response: BinaryIO,
                                         progress_callback: callable
                                         ) -> Tuple[TypeInputFile, int]:
    file_id = helpers.generate_random_long()
    file_size = os.path.getsize(response.name)

    hash_md5 = hashlib.md5()
    uploader = ParallelTransferrer(client)
    part_size, part_count, is_large = await uploader.init_upload(file_id, file_size)
    buffer = bytearray()
    for data in stream_file(response):
        if progress_callback:
            r = progress_callback(response.tell(), file_size)
            if inspect.isawaitable(r):
                await r
        if not is_large:
            hash_md5.update(data)
        if len(buffer) == 0 and len(data) == part_size:
            await uploader.upload(data)
            continue
        new_len = len(buffer) + len(data)
        if new_len >= part_size:
            cutoff = part_size - len(buffer)
            buffer.extend(data[:cutoff])
            await uploader.upload(bytes(buffer))
            buffer.clear()
            buffer.extend(data[cutoff:])
        else:
            buffer.extend(data)
    if len(buffer) > 0:
        await uploader.upload(bytes(buffer))
    await uploader.finish_upload()
    if is_large:
        return InputFileBig(file_id, part_count, "upload"), file_size
    else:
        return InputFile(file_id, part_count, "upload", hash_md5.hexdigest()), file_size


async def download_file(client: TelegramClient,
                        location: TypeLocation,
                        out: BinaryIO,
                        progress_callback: callable = None
                        ) -> BinaryIO:
    size = location.size
    dc_id, location = utils.get_input_location(location)
    # We lock the transfers because telegram has connection count limits
    downloader = ParallelTransferrer(client, dc_id)
    downloaded = downloader.download(location, size)
    async for x in downloaded:
        out.write(x)
        if progress_callback:
            r = progress_callback(out.tell(), size)
            if inspect.isawaitable(r):
                await r

    return out


async def upload_file(client: TelegramClient,
                      file: BinaryIO,
                      progress_callback: callable = None,

                      ) -> TypeInputFile:
    res = (await _internal_transfer_to_telegram(client, file, progress_callback))[0]
    return res


Writing /content/FastTelethon.py


# CODE

In [ ]:
# @title Speed Monitor
import time
import threading
import ipywidgets as widgets
from IPython.display import display, HTML
import datetime

# Function to read current network usage
def get_network_usage():
    try:
        with open('/proc/net/dev', 'r') as f:
            lines = f.readlines()

        for line in lines:
            if 'eth0' in line:  # 'eth0' is the main Colab network interface
                data = line.split()
                download = int(data[1])  # Bytes received
                upload = int(data[9])  # Bytes sent
                return download, upload

        return 0, 0  # Default if no data found
    except Exception as e:
        return 0, 0  # Silently handle errors to avoid bleeding into other cells

# Create dashboard widgets
title = widgets.HTML(value="<h3 style='margin-bottom:10px'>📡 Network Monitor</h3>")
download_text = widgets.HTML(value="⬇️ Download: 0.000 MB/s")
upload_text = widgets.HTML(value="⬆️ Upload: 0.000 MB/s")
timestamp = widgets.HTML(value="Last update: -")
stop_button = widgets.Button(description="Stop", button_style="danger",
                            layout=widgets.Layout(width='80px'))
status_indicator = widgets.HTML(value="<span style='color:green'>●</span> Active")

# Global control variable
monitor_active = {'value': True}

# Update function (to be called from the monitoring thread)
def update_widget_values(download_speed, upload_speed):
    download_text.value = f"⬇️ Download: {download_speed:.3f} MB/s"
    upload_text.value = f"⬆️ Upload: {upload_speed:.3f} MB/s"
    timestamp.value = f"Last update: {datetime.datetime.now().strftime('%H:%M:%S')}"

# The monitoring function that runs in background
def network_monitor_thread(stop_flag):
    prev_download, prev_upload = get_network_usage()

    while stop_flag['value']:
        try:
            time.sleep(1)
            curr_download, curr_upload = get_network_usage()

            # Calculate speeds
            download_speed = (curr_download - prev_download) / 1024 / 1024  # MB/s
            upload_speed = (curr_upload - prev_upload) / 1024 / 1024  # MB/s

            # Update previous values
            prev_download, prev_upload = curr_download, curr_upload

            # Update the widgets (thread-safe operation)
            update_widget_values(download_speed, upload_speed)

        except Exception:
            # Silent exception handling to avoid output bleeding
            pass

# Button click handler
def on_stop_button_clicked(b):
    monitor_active['value'] = False
    status_indicator.value = "<span style='color:red'>●</span> Stopped"
    b.description = "Stopped"
    b.disabled = True

# Start monitoring
def start_network_monitor():
    # Set up the monitoring thread
    stop_button.on_click(on_stop_button_clicked)

    # Create dashboard layout
    dashboard = widgets.VBox([
        title,
        widgets.HBox([download_text, upload_text]),
        widgets.HBox([timestamp, status_indicator, stop_button],
                    layout=widgets.Layout(
                        justify_content='space-between',
                        margin='10px 0 0 0'
                    ))
    ], layout=widgets.Layout(
        border='1px solid #ddd',
        padding='10px',
        margin='10px 0',
        background_color='#f8f8f8',
        width='400px'
    ))

    # Display the dashboard
    display(dashboard)

    # Start the monitoring thread
    thread = threading.Thread(target=network_monitor_thread, args=(monitor_active,))
    thread.daemon = True  # Thread will exit when main program exits
    thread.start()

    # Return a message without polluting output
    return "Network monitor started. You can continue working in other cells."

# Call this function to start monitoring
start_network_monitor()

'Network monitor started. You can continue working in other cells.'

In [ ]:
# @title Upload URLs from CSV/List with Multi-Threaded Downloader revamped thumbnail

# @markdown ## Input Selection
input_type = "CSV File" # @param ["CSV File", "URL List"]

# @markdown ## File Path or URL List
file_path = "/content/direct_streamtape_links_.csv" # @param {type:"string"}

# @markdown ## Upload Settings
include_sender_name = True # @param {type:"boolean"}
thread_count = 4 # @param {type:"slider", min:1, max:8, step:1}

# @markdown ## Account Selection
account_selection = "GP" # @param ["Robi", "GP"]

# @markdown ## Telegram channel ID
channel = "old" # @param ["old","new"]


import logging
import asyncio
import requests
import os
import subprocess
import json
import threading
import urllib.request
import urllib.parse
import re
import csv
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",
        "session_name": "uploader_session_gp"
    }
}


CHANNELS= {
    "old": -1002296651104,
    "new": -1002259201126
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

channel_id = CHANNELS[channel]

# Download settings
NUM_DOWNLOAD_THREADS = thread_count  # Number of threads to use for downloading
DOWNLOAD_FOLDER = 'downloads'  # Folder to store downloaded files

# Option to enable/disable sender name in caption
SHOW_SENDER_NAME = include_sender_name  # Set based on the checkbox value

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


def sanitize_filename(filename):
    """Remove invalid characters from a filename."""
    # Remove invalid characters
    invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    for char in invalid_chars:
        filename = filename.replace(char, '_')

    # Remove any leading/trailing spaces and dots
    filename = filename.strip('. ')

    # Ensure filename is not too long
    if len(filename) > 100:
        name, ext = os.path.splitext(filename)
        filename = name[:95] + ext

    return filename


class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=NUM_DOWNLOAD_THREADS):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        if self.file_size:
            self.chunks = self.get_chunks()
            self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True,
                                desc=f"Downloading {os.path.basename(filename)}")
        else:
            logger.error(f"Could not determine file size for {url}")

    def get_file_size(self):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            req = urllib.request.Request(self.url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req) as response:
                if 'Content-Length' in response.headers:
                    return int(response.headers['Content-Length'])
                return None
        except Exception as e:
            logger.error(f"Error getting file size: {e}")
            return None

    def get_chunks(self):
    # Use a more optimal chunk size (e.g., 8 MB chunks)
      optimal_chunk_size = 32 * 1024 * 1024  # 8 MB
      num_chunks = max(self.num_threads, self.file_size // optimal_chunk_size)
      chunk_size = self.file_size // num_chunks

      chunks = []
      for i in range(num_chunks):
          start = i * chunk_size
          # Important: Make sure there's no gap between chunks
          end = (i + 1) * chunk_size - 1 if i < num_chunks - 1 else self.file_size - 1
          chunks.append((start, end))
      return chunks

    def download_chunk(self, start, end, index):
        try:
            headers = {
                'Range': f"bytes={start}-{end}",
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            req = urllib.request.Request(self.url, headers=headers)
            with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    self.progress.update(len(chunk))
            logger.error(f"Chunk {index} downloaded")
        except Exception as e:
            logger.error(f"Error downloading chunk {index}: {e}")

    def merge_chunks(self):
        try:
            # Verify all part files exist before merging
            missing_parts = []
            for i in range(len(self.chunks)):
                part_filename = f"{self.filename}.part{i}"
                if not os.path.exists(part_filename):
                    missing_parts.append(i)

            if missing_parts:
                logger.error(f"Cannot merge: Missing part files: {missing_parts}")
                self.progress.close()
                return False

            # Check total size of parts matches expected file size
            total_parts_size = sum(os.path.getsize(f"{self.filename}.part{i}")
                                  for i in range(len(self.chunks)))

            if total_parts_size != self.file_size:
                logger.error(f"Size mismatch: Expected {self.file_size} bytes, got {total_parts_size} bytes")
                # Continue anyway, but log the error

            # Merge in correct order
            with open(self.filename, "wb") as final_file:
                for i in range(len(self.chunks)):
                    part_filename = f"{self.filename}.part{i}"
                    with open(part_filename, "rb") as part_file:
                        final_file.write(part_file.read())
                    # Remove part file after successful read
                    os.remove(part_filename)

            self.progress.close()
            logger.error(f"Download completed and merged: {self.filename}")
            return True
        except Exception as e:
            logger.error(f"Error merging chunks: {e}")
            self.progress.close()
            return False

    def verify_download(self):
        """Verify the downloaded file size matches the expected size"""
        if not os.path.exists(self.filename):
            return False

        actual_size = os.path.getsize(self.filename)
        if actual_size != self.file_size:
            logger.error(f"Verification failed: Expected {self.file_size} bytes, got {actual_size} bytes")
            return False
        return True

    def partial_download(self, start=None, end=None):
        """Download a specific part of the file to complete a partial download"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }

            # Add range header if specified
            if start is not None and end is not None:
                headers['Range'] = f"bytes={start}-{end}"
                logger.error(f"Attempting to download missing bytes {start}-{end}")

            req = urllib.request.Request(self.url, headers=headers)

            # Create a temporary file for this partial download
            temp_filename = f"{self.filename}.completion"
            with urllib.request.urlopen(req) as response, open(temp_filename, "wb") as f:
                total_size = int(response.headers.get('Content-Length', 0))
                progress = tqdm(total=total_size, unit='B', unit_scale=True,
                              desc=f"Completing download of {os.path.basename(self.filename)}")

                while True:
                    chunk = response.read(32768)
                    if not chunk:
                        break
                    f.write(chunk)
                    progress.update(len(chunk))

            progress.close()
            return temp_filename

        except Exception as e:
            logger.error(f"Error in partial download: {e}")
            return None

    def fix_download(self):
        """Attempt to fix a corrupted download by identifying and downloading missing parts"""
        try:
            if not os.path.exists(self.filename):
                logger.error("Cannot fix download: File doesn't exist")
                return False

            actual_size = os.path.getsize(self.filename)

            if actual_size > self.file_size:
                # File is larger than expected - truncate it
                logger.warning(f"File is larger than expected ({actual_size} > {self.file_size}). Truncating.")
                with open(self.filename, "r+b") as f:
                    f.truncate(self.file_size)
                return True

            elif actual_size < self.file_size:
                # File is smaller than expected - download the missing part
                logger.warning(f"File is smaller than expected ({actual_size} < {self.file_size}). Downloading missing part.")

                # Download the missing bytes
                temp_file = self.partial_download(start=actual_size, end=self.file_size-1)
                if not temp_file or not os.path.exists(temp_file):
                    return False

                # Append the missing bytes to the original file
                with open(self.filename, "ab") as original, open(temp_file, "rb") as completion:
                    original.write(completion.read())

                # Clean up
                os.remove(temp_file)

                # Verify again
                if os.path.getsize(self.filename) == self.file_size:
                    logger.error("File successfully completed")
                    return True

            return False

        except Exception as e:
            logger.error(f"Error fixing download: {e}")
            return False

    def start(self):
        if not self.file_size:
            return self.fallback_download()

        # Try multi-threaded download
        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        success = self.merge_chunks()

        # Verify and attempt to fix if needed
        if success and not self.verify_download():
            logger.warning("Download verification failed. Attempting to fix...")
            fix_success = self.fix_download()

            if fix_success and self.verify_download():
                logger.error("Download successfully fixed")
                return self.filename
            else:
                logger.warning("Could not fix download. Falling back to single-threaded download")
                if os.path.exists(self.filename):
                    os.remove(self.filename)
                return self.fallback_download()

        return self.filename if success else self.fallback_download()

    def fallback_download(self):
        """Fallback to regular download if multi-threaded download fails"""
        logger.error(f"Using fallback download method for {self.url}")
        try:
            with tqdm(unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(self.filename)}") as progress:
                def report_hook(count, block_size, total_size):
                    if total_size > 0:
                        progress.total = total_size
                        progress.update(block_size)

                headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', headers['User-Agent'])]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.url, self.filename, reporthook=report_hook)
            return self.filename
        except Exception as e:
            logger.error(f"Fallback download failed: {e}")
            return None


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {os.path.basename(self.filename)} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")


async def authenticate():
    try:
        logger.error(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_csv_file(filename=file_path):
    """Read the CSV file and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
            csv_reader = csv.reader(csvfile)
            for row in csv_reader:
                if len(row) >= 2:
                    # First column is filename, second column is URL
                    filename = sanitize_filename(row[0])
                    url = row[1].strip()
                    entries.append((filename, url))

        logger.error(f"Read {len(entries)} entries from CSV file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading CSV file {filename}: {e}")
        return []


async def read_url_list(filename=file_path):
    """Read a text file with URLs and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', encoding='utf-8') as file:
            for line in file:
                url = line.strip()
                if url and not url.startswith('#'):
                    # Extract filename from URL
                    parsed_url = urllib.parse.urlparse(url)
                    path = parsed_url.path
                    filename = os.path.basename(path)

                    # If no filename could be extracted, use the domain with timestamp
                    if not filename or filename == '':
                        domain = parsed_url.netloc.split('.')[-2] if len(parsed_url.netloc.split('.')) > 1 else 'file'
                        timestamp = int(time.time())
                        filename = f"{domain}_{timestamp}.mp4"

                    filename = sanitize_filename(filename)
                    entries.append((filename, url))

        logger.error(f"Read {len(entries)} entries from URL list file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading URL list file {filename}: {e}")
        return []


def check_file_size(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.head(url, headers=headers, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_file(filename, url, download_folder=DOWNLOAD_FOLDER):
    """Download file using multi-threaded downloader with the specified filename"""
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_path = os.path.join(download_folder, filename)

        # Make sure we don't overwrite existing files
        if os.path.exists(file_path):
            base, ext = os.path.splitext(filename)
            timestamp = int(time.time())
            filename = f"{base}_{timestamp}{ext}"
            file_path = os.path.join(download_folder, filename)

        # Use multi-threaded downloader
        downloader = MultiThreadedDownloader(url, file_path, NUM_DOWNLOAD_THREADS)
        result = downloader.start()

        return result
    except Exception as e:
        logger.error(f"Error downloading file from {url}: {e}")
        return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream.get('duration', 0))
            width = int(stream.get('width', 0))
            height = int(stream.get('height', 0))
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


# Ensure the thumbs directory exists
THUMBS_FOLDER = '/content/thumbs'
os.makedirs(THUMBS_FOLDER, exist_ok=True)

def generate_thumbnail(video_path, duration, thumbnail_count=6):
    """
    Generate full-size thumbnails from the video:
    - Generate 6 equal thumbnails.
    - Discard the first thumbnail.
    - Save the remaining 5 thumbnails as PNGs to /content/thumbs.
    """
    thumbnails = []
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return thumbnails

        # Ensure duration is valid
        if not duration or duration <= 0:
            logger.error(f"Invalid duration for video: {duration}")
            return thumbnails

        # Calculate timestamps for 6 equal thumbnails
        interval = duration / thumbnail_count
        timestamps = [i * interval for i in range(thumbnail_count)]

        # Discard the first thumbnail (index 0)
        timestamps = timestamps[1:]

        # Generate thumbnails at calculated timestamps
        for i, timestamp in enumerate(timestamps):
            thumbnail_path = os.path.join(THUMBS_FOLDER, f"thumbnail_{i + 1}.png")
            logger.error(f"Generating thumbnail {i + 1} at {timestamp:.2f}s: {thumbnail_path}")

            # Use ffmpeg to extract a thumbnail
            try:
                command = [
                    'ffmpeg', '-y', '-ss', str(timestamp), '-i', video_path,
                    '-vframes', '1', '-vf', 'scale=iw:-1', thumbnail_path
                ]
                subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
                logger.error(f"Thumbnail generated at {thumbnail_path} using ffmpeg")
                thumbnails.append(thumbnail_path)
            except subprocess.CalledProcessError:
                logger.warning("ffmpeg failed, trying imageio as fallback")

                # Fallback to imageio if ffmpeg fails
                try:
                    video_reader = imageio.get_reader(video_path)
                    fps = video_reader.get_meta_data().get('fps', 30)
                    frame_number = int(timestamp * fps) if fps else 0

                    if frame_number >= len(video_reader):
                        frame_number = 0

                    frame = video_reader.get_data(frame_number)

                    # Convert the frame to an image using Pillow
                    image = Image.fromarray(frame)
                    image = image.resize((image.width, image.height))  # Keep original size

                    # Save the image to the thumbnail file
                    image.save(thumbnail_path)
                    logger.error(f"Thumbnail generated at {thumbnail_path} using imageio")
                    thumbnails.append(thumbnail_path)
                except Exception as e:
                    logger.error(f"Imageio thumbnail generation failed: {e}")
                    continue
    except Exception as e:
        logger.error(f"Error generating thumbnails: {e}")
    return thumbnails

async def send_thumbnails_as_media_group(client, channel_id, thumbnails):
    """
    Send thumbnails as a media group to the specified channel.
    """
    try:
        if thumbnails:
            logger.error("Uploading thumbnails as a media group...")
            await client.send_file(channel_id, thumbnails, grouped=True)
            logger.error("Media group uploaded successfully.")
        else:
            logger.warning("No thumbnails to upload.")
    except Exception as e:
        logger.error(f"Failed to send thumbnails as media group: {e}")

async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Generate 5 full-size thumbnails
        thumbnails = generate_thumbnail(file_path, duration, thumbnail_count=5)

        # Send thumbnails as a media group
        await send_thumbnails_as_media_group(client, channel_id, thumbnails)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=int(duration) if duration else 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time
        caption += f" | Upload Time: {upload_time_formatted}"

        # Add sender name if enabled
        if SHOW_SENDER_NAME:
            caption += f" | {account_selection}"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Send the file without thumbnail (thumbnails already sent)
        await client.send_file(
            entity=channel_id,
            file=media,
            caption=caption
        )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

        # Clean up thumbnails
        for thumbnail in thumbnails:
            if os.path.exists(thumbnail):
                os.remove(thumbnail)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_files():
    await authenticate()

    # Process files based on input type
    if input_type == "CSV File":
        file_entries = await read_csv_file()
    else:  # URL List
        file_entries = await read_url_list()

    for filename, url in file_entries:
        try:
            # Check file size - use HEAD request method
            file_size = check_file_size(url)

            # Skip very large files
            if file_size and file_size > 2000:  # Maximum file size limit of 2000MB (2GB)
                logger.warning(f"Skipping extremely large file: {filename} ({file_size:.2f} MB)")
                continue

            logger.error(f"Processing URL: {url} with filename: {filename}")

            # Download the file using the multi-threaded downloader
            logger.error(f"Starting download of {filename} from {url}")
            downloaded_file = download_file(filename, url)

            if downloaded_file:
                logger.error(f"Successfully downloaded {url} to {downloaded_file}, uploading to Telegram")
                await send_video(client, downloaded_file, channel_id)
            else:
                logger.error(f"Failed to download {url}")
        except Exception as e:
            logger.error(f"Error processing {filename} from {url}: {e}")

if __name__ == '__main__':
    logger.error(f"Starting upload process with {account_selection} account")
    logger.error(f"Using {NUM_DOWNLOAD_THREADS} download threads")
    logger.error(f"Input type: {input_type}")
    logger.error(f"File path: {file_path}")
    logger.error(f"Show sender name: {SHOW_SENDER_NAME}")

    # Create downloads directory if it doesn't exist
    os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_files())

ERROR:__main__:Starting upload process with GP account
ERROR:__main__:Using 4 download threads
ERROR:__main__:Input type: CSV File
ERROR:__main__:File path: /content/direct_streamtape_links_.csv
ERROR:__main__:Show sender name: True
ERROR:__main__:Authenticating with GP account (+8801326573075)
ERROR:__main__:Read 101 entries from CSV file: [JuliaAnnLive] Julia Ann, Cherie Deville (Small Penis Humiliation _ 03.08.2025).mp4
ERROR:__main__:Error checking file size for Direct Link: Invalid URL 'Direct Link': No scheme supplied. Perhaps you meant https://Direct Link?
ERROR:__main__:Processing URL: Direct Link with filename: Name
ERROR:__main__:Starting download of Name from Direct Link
ERROR:__main__:Error getting file size: unknown url type: 'Direct Link'
ERROR:__main__:Could not determine file size for Direct Link
ERROR:__main__:Using fallback download method for Direct Link
ERROR:__main__:Fallback download failed: unknown url type: 'Direct Link'
ERROR:__main__:Failed to download Direct 

KeyboardInterrupt: 

In [ ]:
# @title Upload URLs from CSV/List with Multi-Threaded Downloader revamped thumbnail

# @markdown ## Input Selection
input_type = "CSV File" # @param ["CSV File", "URL List"]

# @markdown ## File Path or URL List
file_path = "/content/direct_streamtape_links_.csv" # @param {type:"string"}

# @markdown ## Upload Settings
include_sender_name = True # @param {type:"boolean"}
thread_count = 4 # @param {type:"slider", min:1, max:8, step:1}

# @markdown ## Account Selection
account_selection = "GP" # @param ["Robi", "GP"]

# @markdown ## Telegram channel ID
channel = "old" # @param ["old","new"]


import logging
import asyncio
import requests
import os
import subprocess
import json
import threading
import urllib.request
import urllib.parse
import re
import csv
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",
        "session_name": "uploader_session_gp"
    }
}


CHANNELS= {
    "old": -1002296651104,
    "new": -1002259201126
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

channel_id = CHANNELS[channel]

# Download settings
NUM_DOWNLOAD_THREADS = thread_count  # Number of threads to use for downloading
DOWNLOAD_FOLDER = 'downloads'  # Folder to store downloaded files

# Option to enable/disable sender name in caption
SHOW_SENDER_NAME = include_sender_name  # Set based on the checkbox value

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


def sanitize_filename(filename):
    """Remove invalid characters from a filename."""
    # Remove invalid characters
    invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    for char in invalid_chars:
        filename = filename.replace(char, '_')

    # Remove any leading/trailing spaces and dots
    filename = filename.strip('. ')

    # Ensure filename is not too long
    if len(filename) > 100:
        name, ext = os.path.splitext(filename)
        filename = name[:95] + ext

    return filename


class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=NUM_DOWNLOAD_THREADS):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        if self.file_size:
            self.chunks = self.get_chunks()
            self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True,
                                desc=f"Downloading {os.path.basename(filename)}")
        else:
            logger.error(f"Could not determine file size for {url}")

    def get_file_size(self):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            req = urllib.request.Request(self.url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req) as response:
                if 'Content-Length' in response.headers:
                    return int(response.headers['Content-Length'])
                return None
        except Exception as e:
            logger.error(f"Error getting file size: {e}")
            return None

    def get_chunks(self):
    # Use a more optimal chunk size (e.g., 8 MB chunks)
      optimal_chunk_size = 32 * 1024 * 1024  # 8 MB
      num_chunks = max(self.num_threads, self.file_size // optimal_chunk_size)
      chunk_size = self.file_size // num_chunks

      chunks = []
      for i in range(num_chunks):
          start = i * chunk_size
          # Important: Make sure there's no gap between chunks
          end = (i + 1) * chunk_size - 1 if i < num_chunks - 1 else self.file_size - 1
          chunks.append((start, end))
      return chunks

    def download_chunk(self, start, end, index):
        try:
            headers = {
                'Range': f"bytes={start}-{end}",
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            req = urllib.request.Request(self.url, headers=headers)
            with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    self.progress.update(len(chunk))
            logger.error(f"Chunk {index} downloaded")
        except Exception as e:
            logger.error(f"Error downloading chunk {index}: {e}")

    def merge_chunks(self):
        try:
            # Verify all part files exist before merging
            missing_parts = []
            for i in range(len(self.chunks)):
                part_filename = f"{self.filename}.part{i}"
                if not os.path.exists(part_filename):
                    missing_parts.append(i)

            if missing_parts:
                logger.error(f"Cannot merge: Missing part files: {missing_parts}")
                self.progress.close()
                return False

            # Check total size of parts matches expected file size
            total_parts_size = sum(os.path.getsize(f"{self.filename}.part{i}")
                                  for i in range(len(self.chunks)))

            if total_parts_size != self.file_size:
                logger.error(f"Size mismatch: Expected {self.file_size} bytes, got {total_parts_size} bytes")
                # Continue anyway, but log the error

            # Merge in correct order
            with open(self.filename, "wb") as final_file:
                for i in range(len(self.chunks)):
                    part_filename = f"{self.filename}.part{i}"
                    with open(part_filename, "rb") as part_file:
                        final_file.write(part_file.read())
                    # Remove part file after successful read
                    os.remove(part_filename)

            self.progress.close()
            logger.error(f"Download completed and merged: {self.filename}")
            return True
        except Exception as e:
            logger.error(f"Error merging chunks: {e}")
            self.progress.close()
            return False

    def verify_download(self):
        """Verify the downloaded file size matches the expected size"""
        if not os.path.exists(self.filename):
            return False

        actual_size = os.path.getsize(self.filename)
        if actual_size != self.file_size:
            logger.error(f"Verification failed: Expected {self.file_size} bytes, got {actual_size} bytes")
            return False
        return True

    def partial_download(self, start=None, end=None):
        """Download a specific part of the file to complete a partial download"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }

            # Add range header if specified
            if start is not None and end is not None:
                headers['Range'] = f"bytes={start}-{end}"
                logger.error(f"Attempting to download missing bytes {start}-{end}")

            req = urllib.request.Request(self.url, headers=headers)

            # Create a temporary file for this partial download
            temp_filename = f"{self.filename}.completion"
            with urllib.request.urlopen(req) as response, open(temp_filename, "wb") as f:
                total_size = int(response.headers.get('Content-Length', 0))
                progress = tqdm(total=total_size, unit='B', unit_scale=True,
                              desc=f"Completing download of {os.path.basename(self.filename)}")

                while True:
                    chunk = response.read(32768)
                    if not chunk:
                        break
                    f.write(chunk)
                    progress.update(len(chunk))

            progress.close()
            return temp_filename

        except Exception as e:
            logger.error(f"Error in partial download: {e}")
            return None

    def fix_download(self):
        """Attempt to fix a corrupted download by identifying and downloading missing parts"""
        try:
            if not os.path.exists(self.filename):
                logger.error("Cannot fix download: File doesn't exist")
                return False

            actual_size = os.path.getsize(self.filename)

            if actual_size > self.file_size:
                # File is larger than expected - truncate it
                logger.warning(f"File is larger than expected ({actual_size} > {self.file_size}). Truncating.")
                with open(self.filename, "r+b") as f:
                    f.truncate(self.file_size)
                return True

            elif actual_size < self.file_size:
                # File is smaller than expected - download the missing part
                logger.warning(f"File is smaller than expected ({actual_size} < {self.file_size}). Downloading missing part.")

                # Download the missing bytes
                temp_file = self.partial_download(start=actual_size, end=self.file_size-1)
                if not temp_file or not os.path.exists(temp_file):
                    return False

                # Append the missing bytes to the original file
                with open(self.filename, "ab") as original, open(temp_file, "rb") as completion:
                    original.write(completion.read())

                # Clean up
                os.remove(temp_file)

                # Verify again
                if os.path.getsize(self.filename) == self.file_size:
                    logger.error("File successfully completed")
                    return True

            return False

        except Exception as e:
            logger.error(f"Error fixing download: {e}")
            return False

    def start(self):
        if not self.file_size:
            return self.fallback_download()

        # Try multi-threaded download
        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        success = self.merge_chunks()

        # Verify and attempt to fix if needed
        if success and not self.verify_download():
            logger.warning("Download verification failed. Attempting to fix...")
            fix_success = self.fix_download()

            if fix_success and self.verify_download():
                logger.error("Download successfully fixed")
                return self.filename
            else:
                logger.warning("Could not fix download. Falling back to single-threaded download")
                if os.path.exists(self.filename):
                    os.remove(self.filename)
                return self.fallback_download()

        return self.filename if success else self.fallback_download()

    def fallback_download(self):
        """Fallback to regular download if multi-threaded download fails"""
        logger.error(f"Using fallback download method for {self.url}")
        try:
            with tqdm(unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(self.filename)}") as progress:
                def report_hook(count, block_size, total_size):
                    if total_size > 0:
                        progress.total = total_size
                        progress.update(block_size)

                headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', headers['User-Agent'])]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.url, self.filename, reporthook=report_hook)
            return self.filename
        except Exception as e:
            logger.error(f"Fallback download failed: {e}")
            return None


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {os.path.basename(self.filename)} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")


async def authenticate():
    try:
        logger.error(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_csv_file(filename=file_path):
    """Read the CSV file and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
            csv_reader = csv.reader(csvfile)
            for row in csv_reader:
                if len(row) >= 2:
                    # First column is filename, second column is URL
                    filename = sanitize_filename(row[0])
                    url = row[1].strip()
                    entries.append((filename, url))

        logger.error(f"Read {len(entries)} entries from CSV file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading CSV file {filename}: {e}")
        return []


async def read_url_list(filename=file_path):
    """Read a text file with URLs and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', encoding='utf-8') as file:
            for line in file:
                url = line.strip()
                if url and not url.startswith('#'):
                    # Extract filename from URL
                    parsed_url = urllib.parse.urlparse(url)
                    path = parsed_url.path
                    filename = os.path.basename(path)

                    # If no filename could be extracted, use the domain with timestamp
                    if not filename or filename == '':
                        domain = parsed_url.netloc.split('.')[-2] if len(parsed_url.netloc.split('.')) > 1 else 'file'
                        timestamp = int(time.time())
                        filename = f"{domain}_{timestamp}.mp4"

                    filename = sanitize_filename(filename)
                    entries.append((filename, url))

        logger.error(f"Read {len(entries)} entries from URL list file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading URL list file {filename}: {e}")
        return []


def check_file_size(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.head(url, headers=headers, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_file(filename, url, download_folder=DOWNLOAD_FOLDER):
    """Download file using multi-threaded downloader with the specified filename"""
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_path = os.path.join(download_folder, filename)

        # Make sure we don't overwrite existing files
        if os.path.exists(file_path):
            base, ext = os.path.splitext(filename)
            timestamp = int(time.time())
            filename = f"{base}_{timestamp}{ext}"
            file_path = os.path.join(download_folder, filename)

        # Use multi-threaded downloader
        downloader = MultiThreadedDownloader(url, file_path, NUM_DOWNLOAD_THREADS)
        result = downloader.start()

        return result
    except Exception as e:
        logger.error(f"Error downloading file from {url}: {e}")
        return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream.get('duration', 0))
            width = int(stream.get('width', 0))
            height = int(stream.get('height', 0))
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


# Ensure the thumbs directory exists
THUMBS_FOLDER = '/content/thumbs'
os.makedirs(THUMBS_FOLDER, exist_ok=True)

def generate_thumbnail(video_path, duration, thumbnail_count=6):
    """
    Generate full-size thumbnails from the video:
    - Generate 6 equal thumbnails.
    - Discard the first thumbnail.
    - Save the remaining 5 thumbnails as PNGs to /content/thumbs.
    """
    thumbnails = []
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return thumbnails

        # Ensure duration is valid
        if not duration or duration <= 0:
            logger.error(f"Invalid duration for video: {duration}")
            return thumbnails

        # Calculate timestamps for 6 equal thumbnails
        interval = duration / thumbnail_count
        timestamps = [i * interval for i in range(thumbnail_count)]

        # Discard the first thumbnail (index 0)
        timestamps = timestamps[1:]

        # Generate thumbnails at calculated timestamps
        for i, timestamp in enumerate(timestamps):
            thumbnail_path = os.path.join(THUMBS_FOLDER, f"thumbnail_{i + 1}.png")
            logger.error(f"Generating thumbnail {i + 1} at {timestamp:.2f}s: {thumbnail_path}")

            # Use ffmpeg to extract a thumbnail
            try:
                command = [
                    'ffmpeg', '-y', '-ss', str(timestamp), '-i', video_path,
                    '-vframes', '1', '-vf', 'scale=iw:-1', thumbnail_path
                ]
                subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
                logger.error(f"Thumbnail generated at {thumbnail_path} using ffmpeg")
                thumbnails.append(thumbnail_path)
            except subprocess.CalledProcessError:
                logger.warning("ffmpeg failed, trying imageio as fallback")

                # Fallback to imageio if ffmpeg fails
                try:
                    video_reader = imageio.get_reader(video_path)
                    fps = video_reader.get_meta_data().get('fps', 30)
                    frame_number = int(timestamp * fps) if fps else 0

                    if frame_number >= len(video_reader):
                        frame_number = 0

                    frame = video_reader.get_data(frame_number)

                    # Convert the frame to an image using Pillow
                    image = Image.fromarray(frame)
                    image = image.resize((image.width, image.height))  # Keep original size

                    # Save the image to the thumbnail file
                    image.save(thumbnail_path)
                    logger.error(f"Thumbnail generated at {thumbnail_path} using imageio")
                    thumbnails.append(thumbnail_path)
                except Exception as e:
                    logger.error(f"Imageio thumbnail generation failed: {e}")
                    continue
    except Exception as e:
        logger.error(f"Error generating thumbnails: {e}")
    return thumbnails

async def send_thumbnails_as_media_group(client, channel_id, thumbnails):
    """
    Send thumbnails as a media group to the specified channel.
    """
    try:
        if thumbnails:
            logger.error("Uploading thumbnails as a media group...")
            await client.send_file(channel_id, thumbnails, grouped=True)
            logger.error("Media group uploaded successfully.")
        else:
            logger.warning("No thumbnails to upload.")
    except Exception as e:
        logger.error(f"Failed to send thumbnails as media group: {e}")

async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Generate 5 full-size thumbnails
        thumbnails = generate_thumbnail(file_path, duration, thumbnail_count=5)

        # Send thumbnails as a media group
        await send_thumbnails_as_media_group(client, channel_id, thumbnails)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=int(duration) if duration else 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time
        caption += f" | Upload Time: {upload_time_formatted}"

        # Add sender name if enabled
        if SHOW_SENDER_NAME:
            caption += f" | {account_selection}"

        # Use the second thumbnail as the video thumbnail
        if len(thumbnails) > 1:
            thumbnail_path = thumbnails[1]  # Second thumbnail (index 1)
            logger.error(f"Using thumbnail {thumbnail_path} as video thumbnail")
            # Upload the thumbnail file to Telegram and get an InputFile object
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await client.upload_file(thumb_file)
        else:
            logger.warning("No second thumbnail available, using default thumbnail")
            thumb = None

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            thumb=thumb,  # Set the thumbnail here
            force_file=False
        )

        # Send the file with the selected thumbnail
        await client.send_file(
            entity=channel_id,
            file=media,
            caption=caption
        )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

        # Clean up thumbnails
        for thumbnail in thumbnails:
            if os.path.exists(thumbnail):
                os.remove(thumbnail)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_files():
    await authenticate()

    # Process files based on input type
    if input_type == "CSV File":
        file_entries = await read_csv_file()
    else:  # URL List
        file_entries = await read_url_list()

    for filename, url in file_entries:
        try:
            # Check file size - use HEAD request method
            file_size = check_file_size(url)

            # Skip very large files
            if file_size and file_size > 2000:  # Maximum file size limit of 2000MB (2GB)
                logger.warning(f"Skipping extremely large file: {filename} ({file_size:.2f} MB)")
                continue

            logger.error(f"Processing URL: {url} with filename: {filename}")

            # Download the file using the multi-threaded downloader
            logger.error(f"Starting download of {filename} from {url}")
            downloaded_file = download_file(filename, url)

            if downloaded_file:
                logger.error(f"Successfully downloaded {url} to {downloaded_file}, uploading to Telegram")
                await send_video(client, downloaded_file, channel_id)
            else:
                logger.error(f"Failed to download {url}")
        except Exception as e:
            logger.error(f"Error processing {filename} from {url}: {e}")

if __name__ == '__main__':
    logger.error(f"Starting upload process with {account_selection} account")
    logger.error(f"Using {NUM_DOWNLOAD_THREADS} download threads")
    logger.error(f"Input type: {input_type}")
    logger.error(f"File path: {file_path}")
    logger.error(f"Show sender name: {SHOW_SENDER_NAME}")

    # Create downloads directory if it doesn't exist
    os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_files())

ERROR:__main__:Starting upload process with GP account
ERROR:__main__:Using 4 download threads
ERROR:__main__:Input type: CSV File
ERROR:__main__:File path: /content/direct_streamtape_links_.csv
ERROR:__main__:Show sender name: True
ERROR:__main__:Authenticating with GP account (+8801326573075)
ERROR:__main__:Read 91 entries from CSV file: [JuliaAnnLive] Julia Ann, Cherie Deville (Small Penis Humiliation _ 03.08.2025).mp4
ERROR:__main__:Error checking file size for Direct Link: Invalid URL 'Direct Link': No scheme supplied. Perhaps you meant https://Direct Link?
ERROR:__main__:Processing URL: Direct Link with filename: Name
ERROR:__main__:Starting download of Name from Direct Link
ERROR:__main__:Error getting file size: unknown url type: 'Direct Link'
ERROR:__main__:Could not determine file size for Direct Link
ERROR:__main__:Using fallback download method for Direct Link
ERROR:__main__:Fallback download failed: unknown url type: 'Direct Link'
ERROR:__main__:Failed to download Direct L

In [ ]:
#@title LINK to CSV
import requests
import re
import csv
from time import sleep
from bs4 import BeautifulSoup

# Define the URL list file
URL_LIST = '/content/x.txt'

# Define the output file for direct download links
CSV_OUTPUT_FILE = "direct_streamtape_links.csv"

# Define the maximum number of retries for 502 errors
MAX_RETRIES = 3

# Define the delay between retries (in seconds)
RETRY_DELAY = 5

# Regular expression to match Streamtape links
STREAMTAPE_REGEX = re.compile(r"https://streamtape\.[a-z]+/[^\s'\"]+")

def fetch_url(url, retries=MAX_RETRIES):
    """
    Fetch the content of a URL with retries for HTTP 502 errors.
    """
    for attempt in range(retries):
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for HTTP errors
            return response.text
        except requests.exceptions.HTTPError as e:
            if response.status_code == 502 and attempt < retries - 1:
                print(f"[ERROR] Failed to fetch {url} (HTTP 502). Retrying in {RETRY_DELAY} seconds...")
                sleep(RETRY_DELAY)
            else:
                print(f"[ERROR] Failed to fetch {url} after {retries} attempts: {e}")
                return None
    return None

def extract_streamtape_links(html):
    """
    Extract Streamtape links from HTML content.
    """
    return STREAMTAPE_REGEX.findall(html)

def extract_title(html):
    """
    Extract the title from the HTML content.
    """
    try:
        soup = BeautifulSoup(html, 'html.parser')
        header = soup.select_one('header.entry-header')
        if header:
            title_element = header.select_one('h1.entry-title')
            if title_element:
                return title_element.text.strip()
    except Exception as e:
        print(f"[ERROR] Failed to extract title: {e}")

    return "Unknown Title"

def steamtape_get_dl_link(link):
    try:
        if "/e/" in link:
            link = link.replace("/e/", "/v/")

        response = requests.get(link)
        response.raise_for_status()
        html_source = response.text

        norobot_link_pattern = re.compile(r"document\.getElementById\('norobotlink'\)\.innerHTML = (.+?);")
        norobot_link_matcher = norobot_link_pattern.search(html_source)

        if norobot_link_matcher:
            norobot_link_content = norobot_link_matcher.group(1)

            token_pattern = re.compile(r"token=([^&']+)")
            token_matcher = token_pattern.search(norobot_link_content)

            if token_matcher:
                token = token_matcher.group(1)

                soup = BeautifulSoup(html_source, 'html.parser')
                div_element = soup.select_one("div#ideoooolink[style='display:none;']")

                if div_element:
                    streamtape = div_element.get_text()
                    full_url = f"https:/{streamtape}&token={token}"
                    return f"{full_url}&dl=1s"

    except Exception as exception:
        print(f"An error occurred: {exception}")

    return None

def main():
    # Read the list of URLs
    try:
        with open(URL_LIST, "r") as file:
            urls = [line.strip() for line in file if line.strip()]
    except FileNotFoundError:
        print(f"[ERROR] URL list file not found: {URL_LIST}")
        return

    # Open the CSV output file for direct download links with titles
    with open(CSV_OUTPUT_FILE, "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file)

        # Write the header row
        csv_writer.writerow(["Name", "Direct Link"])

        print("[INFO] Starting Streamtape link extraction...")

        # Process each URL
        for url in urls:
            print(f"[INFO] Fetching: {url}")
            html = fetch_url(url)

            if html:
                # Extract title from the page
                title = extract_title(html)
                print(f"[INFO] Found title: {title}")

                # Add mp4 extension to the title
                title_with_extension = f"{title}.mp4"

                # Extract Streamtape links
                streamtape_links = extract_streamtape_links(html)
                if streamtape_links:
                    for link in streamtape_links:
                        print(f"[INFO] Found Streamtape link: {link}")

                        # Get the direct download link for each Streamtape link
                        direct_link = steamtape_get_dl_link(link)
                        if direct_link:
                            print(f"[INFO] Found direct download link: {direct_link}")

                            # Write the title and direct link to the CSV file
                            csv_writer.writerow([title_with_extension, direct_link])
                        else:
                            print(f"[WARNING] Could not find direct download link for {link}")
                else:
                    print(f"[WARNING] No Streamtape links found in {url}")
            else:
                print(f"[ERROR] Skipping {url} due to fetch failure")

        print(f"[INFO] All direct download links with titles saved in {CSV_OUTPUT_FILE}.")

if __name__ == "__main__":
    main()

[INFO] Starting Streamtape link extraction...
[INFO] Fetching: https://xmoviesforyou.com/2025/03/assparade-nuria-millan-2-big-cocks-for-nuria.html
[INFO] Found title: [AssParade] Nuria Millan (2 Big Cocks For Nuria / 03.10.2025)
[INFO] Found Streamtape link: https://streamtape.to/e/2DKZM8od9XhZ2V4
[INFO] Found direct download link: https://streamtape.to/get_video?id=2DKZM8od9XhZ2V4&expires=1741718672&ip=FHEsFxSnDS9X&token=WTllzT6wVvzZ&token=WTllzT6wVvBj&dl=1s
[INFO] Fetching: https://xmoviesforyou.com/2025/03/mylfsingles-sarah-jessie-hyley-winters-swiss-milk-to-seal-the-deal.html
[INFO] Found title: [MylfSingles] Sarah Jessie, Hyley Winters (Swiss Milk to Seal the Deal / 03.10.2025)
[INFO] Found Streamtape link: https://streamtape.to/e/WDRQZ3O7avTbd4D
[INFO] Found direct download link: https://streamtape.to/get_video?id=WDRQZ3O7avTbd4D&expires=1741718674&ip=FHEsFxSnDS9X&token=7ISagJmiEIzZ&token=7ISagJmiEIOn&dl=1s
[INFO] Fetching: https://xmoviesforyou.com/2025/03/digitalplayground-litt

KeyboardInterrupt: 

In [ ]:
import csv

input_file = "/content/direct_streamtape_links.csv"  # Change this to your CSV file name
output_file = "/content/direct_streamtape_links_.csv"

with open(input_file, "r", newline="", encoding="utf-8") as infile, open(output_file, "w", newline="", encoding="utf-8") as outfile:
    reader = csv.DictReader(infile)
    fieldnames = reader.fieldnames

    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    for row in reader:
        if "Name" in row:  # Ensure the column exists
            row["Name"] = row["Name"].replace("/", "_")
        writer.writerow(row)

print(f"File processed. Output saved as {output_file}")


File processed. Output saved as /content/direct_streamtape_links_.csv


# With thumb and old files



In [ ]:
# @title 4. Upload url with thumbnail

# @markdown ## Account Selection
# @markdown Select which account to use for uploading
account_selection = "GP" # @param ["Robi", "GP"]

import logging
import asyncio
import requests
import os
import subprocess
import json
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",  # Update with correct GP phone number
        "session_name": "uploader_session_gp"
    }
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

# Telegram channel ID
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/url.txt'

# Option to enable/disable upload time in caption
SHOW_UPLOAD_TIME = True  # Set to False to disable showing upload time in caption

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {self.filename} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")

async def authenticate():
    try:
        logger.info(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=128 * 1024):  # Increased chunk size
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream['duration'])
            width = int(stream['width'])
            height = int(stream['height'])
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1

        # Use imageio to read the video frame at the given timestamp
        video_reader = imageio.get_reader(video_path)
        frame = video_reader.get_data(int(timestamp * video_reader.get_meta_data()['fps']))

        # Convert the frame to an image using Pillow
        image = Image.fromarray(frame)
        image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

        # Save the image to the temporary thumbnail file
        image.save(thumbnail_path)
        logger.info(f"Thumbnail generated at {thumbnail_path}")
        return thumbnail_path
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=duration or 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time if enabled
        if SHOW_UPLOAD_TIME:
            caption += f" | Upload Time: {upload_time_formatted}"

        caption += f" | {account_selection}"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        file_size = check_file_size(url)

        # Skip very large files
        if file_size and file_size > 200:  # Increased file size limit
            logger.warning(f"Skipping extremely large file: {url}")
            continue

        video_file = download_video(url)

        if video_file:
            await send_video(client, video_file, channel_id)

# Run the async function
if __name__ == '__main__':
    logger.info(f"Starting upload process with {account_selection} account")
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())

In [ ]:
# @title 4. Upload url with thumbnail with Multi-Threaded Downloader v1

# @markdown ## Account Selection
# @markdown Select which account to use for uploading
account_selection = "GP" # @param ["Robi", "GP"]

import logging
import asyncio
import requests
import os
import subprocess
import json
import threading
import urllib.request
import urllib.parse
import re
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",
        "session_name": "uploader_session_gp"
    }
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

# Telegram channel ID
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/direct_streamtape_links.txt'

# Download settings
NUM_DOWNLOAD_THREADS = 4  # Number of threads to use for downloading
DOWNLOAD_FOLDER = 'downloads'  # Folder to store downloaded files

# Option to enable/disable upload time in caption
SHOW_UPLOAD_TIME = True  # Set to False to disable showing upload time in caption

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


def extract_filename_from_url(url):
    """Extract a proper filename from the URL."""
    try:
        # Try to get filename from Content-Disposition header
        response = requests.head(url, allow_redirects=True)
        if 'Content-Disposition' in response.headers:
            content_disposition = response.headers['Content-Disposition']
            filename_match = re.search(r'filename="?([^";]+)"?', content_disposition)
            if filename_match:
                return filename_match.group(1)

        # Parse the URL path
        parsed_url = urllib.parse.urlparse(url)
        path = parsed_url.path

        # Remove query parameters from the path
        path = path.split('?')[0]

        # Get the last segment of the path (usually the filename)
        filename = os.path.basename(path)

        # If filename is empty or just a trailing slash
        if not filename or filename == '/':
            # Create a filename from the domain with a timestamp
            domain = parsed_url.netloc.split('.')[-2]  # Get the domain name
            timestamp = int(time.time())
            extension = guess_extension_from_url(url) or '.mp4'  # Default to .mp4 for video downloads
            filename = f"{domain}_{timestamp}{extension}"

        # Make sure it doesn't have problematic characters
        filename = sanitize_filename(filename)

        # If we end up with an empty filename or just a dot
        if not filename or filename == '.':
            # Generate a random filename
            timestamp = int(time.time())
            filename = f"download_{timestamp}.mp4"

        return filename
    except Exception as e:
        logger.error(f"Error extracting filename from URL: {e}")
        # Return a fallback filename
        timestamp = int(time.time())
        return f"download_{timestamp}.mp4"


def guess_extension_from_url(url):
    """Try to guess the file extension from the URL or mime type."""
    try:
        # Try to get Content-Type
        response = requests.head(url, allow_redirects=True)
        content_type = response.headers.get('Content-Type', '')

        # Map common content types to extensions
        extension_map = {
            'video/mp4': '.mp4',
            'video/webm': '.webm',
            'video/x-matroska': '.mkv',
            'video/quicktime': '.mov',
            'video/x-msvideo': '.avi',
            'video/x-flv': '.flv',
            'video/x-ms-wmv': '.wmv',
            'video/mpeg': '.mpeg',
            'application/octet-stream': '.bin'
        }

        # Check if we have a matching content type
        if content_type in extension_map:
            return extension_map[content_type]

        # Otherwise try to extract extension from URL path
        parsed_url = urllib.parse.urlparse(url)
        path = parsed_url.path
        extension = os.path.splitext(path)[1]

        if extension:
            return extension

        # Try to extract extension from query parameters
        if 'id' in parsed_url.query and '.mp4' not in path:
            return '.mp4'  # Common for video streaming sites

        return ''
    except Exception as e:
        logger.error(f"Error guessing extension: {e}")
        return ''


def sanitize_filename(filename):
    """Remove invalid characters from a filename."""
    # Remove invalid characters
    invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    for char in invalid_chars:
        filename = filename.replace(char, '_')

    # Remove any leading/trailing spaces and dots
    filename = filename.strip('. ')

    # Ensure filename is not too long
    if len(filename) > 100:
        name, ext = os.path.splitext(filename)
        filename = name[:95] + ext

    return filename


class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=NUM_DOWNLOAD_THREADS):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        if self.file_size:
            self.chunks = self.get_chunks()
            self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True,
                                desc=f"Downloading {os.path.basename(filename)}")
        else:
            logger.error(f"Could not determine file size for {url}")

    def get_file_size(self):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            req = urllib.request.Request(self.url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req) as response:
                if 'Content-Length' in response.headers:
                    return int(response.headers['Content-Length'])
                return None
        except Exception as e:
            logger.error(f"Error getting file size: {e}")
            return None

    def get_chunks(self):
        chunk_size = self.file_size // self.num_threads
        chunks = []
        for i in range(self.num_threads):
            start = i * chunk_size
            end = (start + chunk_size - 1) if i < self.num_threads - 1 else self.file_size - 1
            chunks.append((start, end))
        return chunks

    def download_chunk(self, start, end, index):
        try:
            headers = {
                'Range': f"bytes={start}-{end}",
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            req = urllib.request.Request(self.url, headers=headers)
            with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    self.progress.update(len(chunk))
            logger.info(f"Chunk {index} downloaded")
        except Exception as e:
            logger.error(f"Error downloading chunk {index}: {e}")

    def merge_chunks(self):
        try:
            with open(self.filename, "wb") as final_file:
                for i in range(self.num_threads):
                    part_filename = f"{self.filename}.part{i}"
                    if os.path.exists(part_filename):
                        with open(part_filename, "rb") as part_file:
                            final_file.write(part_file.read())
                        os.remove(part_filename)
                    else:
                        logger.warning(f"Part file missing: {part_filename}")
            self.progress.close()
            logger.info(f"Download completed and merged: {self.filename}")
            return True
        except Exception as e:
            logger.error(f"Error merging chunks: {e}")
            self.progress.close()
            return False

    def start(self):
        if not self.file_size:
            return self.fallback_download()

        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        success = self.merge_chunks()
        return self.filename if success else None

    def fallback_download(self):
        """Fallback to regular download if multi-threaded download fails"""
        logger.info(f"Using fallback download method for {self.url}")
        try:
            with tqdm(unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(self.filename)}") as progress:
                def report_hook(count, block_size, total_size):
                    if total_size > 0:
                        progress.total = total_size
                        progress.update(block_size)

                headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', headers['User-Agent'])]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.url, self.filename, reporthook=report_hook)
            return self.filename
        except Exception as e:
            logger.error(f"Fallback download failed: {e}")
            return None


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {os.path.basename(self.filename)} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")


async def authenticate():
    try:
        logger.info(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.head(url, headers=headers, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder=DOWNLOAD_FOLDER):
    """Download video using multi-threaded downloader with proper filename extraction"""
    try:
        os.makedirs(download_folder, exist_ok=True)

        # Extract a proper filename from the URL
        filename = extract_filename_from_url(url)
        logger.info(f"Extracted filename: {filename} from URL: {url}")

        # Make sure the filename has a video extension
        if not os.path.splitext(filename)[1]:
            filename += '.mp4'

        file_path = os.path.join(download_folder, filename)

        # Make sure we don't overwrite existing files
        if os.path.exists(file_path):
            base, ext = os.path.splitext(filename)
            timestamp = int(time.time())
            filename = f"{base}_{timestamp}{ext}"
            file_path = os.path.join(download_folder, filename)

        # Use multi-threaded downloader
        downloader = MultiThreadedDownloader(url, file_path, NUM_DOWNLOAD_THREADS)
        result = downloader.start()

        return result
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
        return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream.get('duration', 0))
            width = int(stream.get('width', 0))
            height = int(stream.get('height', 0))
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1 if duration else 5  # Default to 5 seconds if duration unknown

        # Use ffmpeg to extract a thumbnail
        try:
            command = [
                'ffmpeg', '-y', '-ss', str(timestamp), '-i', video_path,
                '-vframes', '1', '-vf', 'scale=320:-1', thumbnail_path
            ]
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            logger.info(f"Thumbnail generated at {thumbnail_path} using ffmpeg")
            return thumbnail_path
        except subprocess.CalledProcessError:
            logger.warning("ffmpeg failed, trying imageio as fallback")

            # Fallback to imageio if ffmpeg fails
            try:
                video_reader = imageio.get_reader(video_path)
                fps = video_reader.get_meta_data().get('fps', 30)
                frame_number = int(timestamp * fps) if fps else 0

                if frame_number >= len(video_reader):
                    frame_number = 0

                frame = video_reader.get_data(frame_number)

                # Convert the frame to an image using Pillow
                image = Image.fromarray(frame)
                image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

                # Save the image to the temporary thumbnail file
                image.save(thumbnail_path)
                logger.info(f"Thumbnail generated at {thumbnail_path} using imageio")
                return thumbnail_path
            except Exception as e:
                logger.error(f"Imageio thumbnail generation failed: {e}")
                return None
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=int(duration) if duration else 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time if enabled
        if SHOW_UPLOAD_TIME:
            caption += f" | Upload Time: {upload_time_formatted}"

        caption += f" | {account_selection}"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        try:
            # Check file size - use HEAD request method
            file_size = check_file_size(url)

            # Skip very large files
            if file_size and file_size > 2000:  # Increased file size limit to 200MB
                logger.warning(f"Skipping extremely large file: {url} ({file_size:.2f} MB)")
                continue

            # Extract filename first for logging purposes
            filename = extract_filename_from_url(url)
            logger.info(f"Processing URL: {url} with detected filename: {filename}")

            # Download the video using the multi-threaded downloader
            logger.info(f"Starting download of {filename} from {url}")
            video_file = download_video(url)

            if video_file:
                logger.info(f"Successfully downloaded {url} to {video_file}, uploading to Telegram")
                await send_video(client, video_file, channel_id)
            else:
                logger.error(f"Failed to download {url}")
        except Exception as e:
            logger.error(f"Error processing URL {url}: {e}")

# Run the async function
if __name__ == '__main__':
    logger.info(f"Starting upload process with {account_selection} account")
    # Create downloads directory if it doesn't exist
    os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())

In [ ]:
# @title Upload URLs from CSV with Multi-Threaded Downloader

# @markdown ## Account Selection
# @markdown Select which account to use for uploading
account_selection = "GP" # @param ["Robi", "GP"]

# @markdown ## CSV File Path
# @markdown Enter the path to your CSV file
csv_file_path = "/content/direct_streamtape_links.csv" # @param {type:"string"}

import logging
import asyncio
import requests
import os
import subprocess
import json
import threading
import urllib.request
import urllib.parse
import re
import csv
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",
        "session_name": "uploader_session_gp"
    }
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

# Telegram channel ID
channel_id = -1002296651104  # Telegram channel ID

# Download settings
NUM_DOWNLOAD_THREADS = 2  # Number of threads to use for downloading
DOWNLOAD_FOLDER = 'downloads'  # Folder to store downloaded files

# Option to enable/disable upload time in caption
SHOW_UPLOAD_TIME = True  # Set to False to disable showing upload time in caption

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


def sanitize_filename(filename):
    """Remove invalid characters from a filename."""
    # Remove invalid characters
    invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    for char in invalid_chars:
        filename = filename.replace(char, '_')

    # Remove any leading/trailing spaces and dots
    filename = filename.strip('. ')

    # Ensure filename is not too long
    if len(filename) > 100:
        name, ext = os.path.splitext(filename)
        filename = name[:95] + ext

    return filename


class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=NUM_DOWNLOAD_THREADS):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        if self.file_size:
            self.chunks = self.get_chunks()
            self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True,
                                desc=f"Downloading {os.path.basename(filename)}")
        else:
            logger.error(f"Could not determine file size for {url}")

    def get_file_size(self):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            req = urllib.request.Request(self.url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req) as response:
                if 'Content-Length' in response.headers:
                    return int(response.headers['Content-Length'])
                return None
        except Exception as e:
            logger.error(f"Error getting file size: {e}")
            return None

    def get_chunks(self):
    # Use a more optimal chunk size (e.g., 8 MB chunks)
      optimal_chunk_size = 32 * 1024 * 1024  # 8 MB
      num_chunks = max(self.num_threads, self.file_size // optimal_chunk_size)
      chunk_size = self.file_size // num_chunks

      chunks = []
      for i in range(num_chunks):
          start = i * chunk_size
          # Important: Make sure there's no gap between chunks
          end = (i + 1) * chunk_size - 1 if i < num_chunks - 1 else self.file_size - 1
          chunks.append((start, end))
      return chunks

    def download_chunk(self, start, end, index):
        try:
            headers = {
                'Range': f"bytes={start}-{end}",
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            req = urllib.request.Request(self.url, headers=headers)
            with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    self.progress.update(len(chunk))
            logger.info(f"Chunk {index} downloaded")
        except Exception as e:
            logger.error(f"Error downloading chunk {index}: {e}")

    def merge_chunks(self):
        try:
            # Verify all part files exist before merging
            missing_parts = []
            for i in range(len(self.chunks)):
                part_filename = f"{self.filename}.part{i}"
                if not os.path.exists(part_filename):
                    missing_parts.append(i)

            if missing_parts:
                logger.error(f"Cannot merge: Missing part files: {missing_parts}")
                self.progress.close()
                return False

            # Check total size of parts matches expected file size
            total_parts_size = sum(os.path.getsize(f"{self.filename}.part{i}")
                                  for i in range(len(self.chunks)))

            if total_parts_size != self.file_size:
                logger.error(f"Size mismatch: Expected {self.file_size} bytes, got {total_parts_size} bytes")
                # Continue anyway, but log the error

            # Merge in correct order
            with open(self.filename, "wb") as final_file:
                for i in range(len(self.chunks)):
                    part_filename = f"{self.filename}.part{i}"
                    with open(part_filename, "rb") as part_file:
                        final_file.write(part_file.read())
                    # Remove part file after successful read
                    os.remove(part_filename)

            self.progress.close()
            logger.info(f"Download completed and merged: {self.filename}")
            return True
        except Exception as e:
            logger.error(f"Error merging chunks: {e}")
            self.progress.close()
            return False

    def verify_download(self):
        """Verify the downloaded file size matches the expected size"""
        if not os.path.exists(self.filename):
            return False

        actual_size = os.path.getsize(self.filename)
        if actual_size != self.file_size:
            logger.error(f"Verification failed: Expected {self.file_size} bytes, got {actual_size} bytes")
            return False
        return True

    def partial_download(self, start=None, end=None):
        """Download a specific part of the file to complete a partial download"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }

            # Add range header if specified
            if start is not None and end is not None:
                headers['Range'] = f"bytes={start}-{end}"
                logger.info(f"Attempting to download missing bytes {start}-{end}")

            req = urllib.request.Request(self.url, headers=headers)

            # Create a temporary file for this partial download
            temp_filename = f"{self.filename}.completion"
            with urllib.request.urlopen(req) as response, open(temp_filename, "wb") as f:
                total_size = int(response.headers.get('Content-Length', 0))
                progress = tqdm(total=total_size, unit='B', unit_scale=True,
                              desc=f"Completing download of {os.path.basename(self.filename)}")

                while True:
                    chunk = response.read(32768)
                    if not chunk:
                        break
                    f.write(chunk)
                    progress.update(len(chunk))

            progress.close()
            return temp_filename

        except Exception as e:
            logger.error(f"Error in partial download: {e}")
            return None

    def fix_download(self):
        """Attempt to fix a corrupted download by identifying and downloading missing parts"""
        try:
            if not os.path.exists(self.filename):
                logger.error("Cannot fix download: File doesn't exist")
                return False

            actual_size = os.path.getsize(self.filename)

            if actual_size > self.file_size:
                # File is larger than expected - truncate it
                logger.warning(f"File is larger than expected ({actual_size} > {self.file_size}). Truncating.")
                with open(self.filename, "r+b") as f:
                    f.truncate(self.file_size)
                return True

            elif actual_size < self.file_size:
                # File is smaller than expected - download the missing part
                logger.warning(f"File is smaller than expected ({actual_size} < {self.file_size}). Downloading missing part.")

                # Download the missing bytes
                temp_file = self.partial_download(start=actual_size, end=self.file_size-1)
                if not temp_file or not os.path.exists(temp_file):
                    return False

                # Append the missing bytes to the original file
                with open(self.filename, "ab") as original, open(temp_file, "rb") as completion:
                    original.write(completion.read())

                # Clean up
                os.remove(temp_file)

                # Verify again
                if os.path.getsize(self.filename) == self.file_size:
                    logger.info("File successfully completed")
                    return True

            return False

        except Exception as e:
            logger.error(f"Error fixing download: {e}")
            return False

    def start(self):
        if not self.file_size:
            return self.fallback_download()

        # Try multi-threaded download
        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        success = self.merge_chunks()

        # Verify and attempt to fix if needed
        if success and not self.verify_download():
            logger.warning("Download verification failed. Attempting to fix...")
            fix_success = self.fix_download()

            if fix_success and self.verify_download():
                logger.info("Download successfully fixed")
                return self.filename
            else:
                logger.warning("Could not fix download. Falling back to single-threaded download")
                if os.path.exists(self.filename):
                    os.remove(self.filename)
                return self.fallback_download()

        return self.filename if success else self.fallback_download()

    def fallback_download(self):
        """Fallback to regular download if multi-threaded download fails"""
        logger.info(f"Using fallback download method for {self.url}")
        try:
            with tqdm(unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(self.filename)}") as progress:
                def report_hook(count, block_size, total_size):
                    if total_size > 0:
                        progress.total = total_size
                        progress.update(block_size)

                headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', headers['User-Agent'])]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.url, self.filename, reporthook=report_hook)
            return self.filename
        except Exception as e:
            logger.error(f"Fallback download failed: {e}")
            return None


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {os.path.basename(self.filename)} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")


async def authenticate():
    try:
        logger.info(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_csv_file(filename=csv_file_path):
    """Read the CSV file and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
            csv_reader = csv.reader(csvfile)
            for row in csv_reader:
                if len(row) >= 2:
                    # First column is filename, second column is URL
                    filename = sanitize_filename(row[0])
                    url = row[1].strip()
                    entries.append((filename, url))

        logger.info(f"Read {len(entries)} entries from CSV file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading CSV file {filename}: {e}")
        return []


def check_file_size(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.head(url, headers=headers, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_file(filename, url, download_folder=DOWNLOAD_FOLDER):
    """Download file using multi-threaded downloader with the specified filename"""
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_path = os.path.join(download_folder, filename)

        # Make sure we don't overwrite existing files
        if os.path.exists(file_path):
            base, ext = os.path.splitext(filename)
            timestamp = int(time.time())
            filename = f"{base}_{timestamp}{ext}"
            file_path = os.path.join(download_folder, filename)

        # Use multi-threaded downloader
        downloader = MultiThreadedDownloader(url, file_path, NUM_DOWNLOAD_THREADS)
        result = downloader.start()

        return result
    except Exception as e:
        logger.error(f"Error downloading file from {url}: {e}")
        return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream.get('duration', 0))
            width = int(stream.get('width', 0))
            height = int(stream.get('height', 0))
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1 if duration else 5  # Default to 5 seconds if duration unknown

        # Use ffmpeg to extract a thumbnail
        try:
            command = [
                'ffmpeg', '-y', '-ss', str(timestamp), '-i', video_path,
                '-vframes', '1', '-vf', 'scale=320:-1', thumbnail_path
            ]
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            logger.info(f"Thumbnail generated at {thumbnail_path} using ffmpeg")
            return thumbnail_path
        except subprocess.CalledProcessError:
            logger.warning("ffmpeg failed, trying imageio as fallback")

            # Fallback to imageio if ffmpeg fails
            try:
                video_reader = imageio.get_reader(video_path)
                fps = video_reader.get_meta_data().get('fps', 30)
                frame_number = int(timestamp * fps) if fps else 0

                if frame_number >= len(video_reader):
                    frame_number = 0

                frame = video_reader.get_data(frame_number)

                # Convert the frame to an image using Pillow
                image = Image.fromarray(frame)
                image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

                # Save the image to the temporary thumbnail file
                image.save(thumbnail_path)
                logger.info(f"Thumbnail generated at {thumbnail_path} using imageio")
                return thumbnail_path
            except Exception as e:
                logger.error(f"Imageio thumbnail generation failed: {e}")
                return None
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=int(duration) if duration else 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time if enabled
        if SHOW_UPLOAD_TIME:
            caption += f" | Upload Time: {upload_time_formatted}"

        caption += f" | {account_selection}"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_files():
    await authenticate()
    file_entries = await read_csv_file()

    for filename, url in file_entries:
        try:
            # Check file size - use HEAD request method
            file_size = check_file_size(url)

            # Skip very large files
            if file_size and file_size > 2000:  # Maximum file size limit of 2000MB (2GB)
                logger.warning(f"Skipping extremely large file: {filename} ({file_size:.2f} MB)")
                continue

            logger.info(f"Processing URL: {url} with filename: {filename}")

            # Download the file using the multi-threaded downloader
            logger.info(f"Starting download of {filename} from {url}")
            downloaded_file = download_file(filename, url)

            if downloaded_file:
                logger.info(f"Successfully downloaded {url} to {downloaded_file}, uploading to Telegram")
                await send_video(client, downloaded_file, channel_id)
            else:
                logger.error(f"Failed to download {url}")
        except Exception as e:
            logger.error(f"Error processing {filename} from {url}: {e}")

# Run the async function
if __name__ == '__main__':
    logger.info(f"Starting upload process with {account_selection} account")
    # Create downloads directory if it doesn't exist
    os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_files())

ERROR:__main__:Error checking file size for Direct Link: Invalid URL 'Direct Link': No scheme supplied. Perhaps you meant https://Direct Link?
ERROR:__main__:Error getting file size: unknown url type: 'Direct Link'
ERROR:__main__:Could not determine file size for Direct Link
ERROR:__main__:Fallback download failed: unknown url type: 'Direct Link'
ERROR:__main__:Failed to download Direct Link
ERROR:__main__:Error checking file size for https://streamtape.to/get_video?id=K0Ree3lDRDc0OPZ&expires=1741465955&ip=FHIsD0cNKxSHDN&token=2W1R_cf2GbzZ&token=2W1R_cf2Gbzv&dl=1s: HTTPSConnectionPool(host='2565136607.tapecontent.net', port=443): Max retries exceeded with url: /radosgw/K0Ree3lDRDc0OPZ/NUqQ1716iWP_Uc5MMotC_GHOQEajpLJe5vnivrumon3YNO0q2Um7YHCau4pdoArg48yKC2tTzI6X470EkeY7El9uMVYS3OlnBAxD2qf4UAHkG8P7jth8Rs6sCT49tqPrGzMH_KN1Zb5CqRUChTNrOcEzEG7KXghL5w8LUpax_1X_ZEGgAVns9Flfe8an6NSPGo2Tw1cEkLwRszi_X7aMwzfwihgZB9lpE-HMLcueHcxNxuMlPLhweltyUUsOh9g1msLtb-afvJYGEZUZUwMx22lEWzBYp0udp1PsNL0ToJMz-vzKU7

In [ ]:
import logging
import asyncio
import requests
import os
import subprocess
import json
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Telegram Bot and Telethon credentials
api_id = 1233728  # Get from https://my.telegram.org
api_hash = '0959897ff7f45100d243daa78437fb7b'  # Get from https://my.telegram.org
phone_number = '+8801828767185'  # Your phone number with country code
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/url.txt'

# Initialize Telethon client
client = TelegramClient('uploader_session', api_id, api_hash)


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"COLAB Uploading {self.filename} [{current / (1024 * 1024):.2f}/{self.formatted_file_size}] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")

async def authenticate():
    try:
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=128 * 1024):  # Increased chunk size
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream['duration'])
            width = int(stream['width'])
            height = int(stream['height'])
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload COLAB {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=duration or 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create caption with file name and size
        caption = f"COLAB {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Send the file
        await client.send_file(
            entity=channel_id,
            file=media,
            caption=caption
        )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        file_size = check_file_size(url)

        # Skip very large files
        if file_size and file_size > 200:  # Increased file size limit
            logger.warning(f"Skipping extremely large file: {url}")
            continue

        video_file = download_video(url)

        if video_file:
            await send_video(client, video_file, channel_id)

# Run the async function
if __name__ == '__main__':
    nest_asyncio.apply() # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())


In [ ]:
# @title 4. Upload url with thumbnail extra detail
import logging
import asyncio
import requests
import os
import subprocess
import json
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Telegram Bot and Telethon credentials
api_id = 1233728  # Get from https://my.telegram.org
api_hash = '0959897ff7f45100d243daa78437fb7b'  # Get from https://my.telegram.org
phone_number = '+8801828767185'  # Your phone number with country code
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/url.txt'

# Option to enable/disable upload time in caption
SHOW_UPLOAD_TIME = True  # Set to False to disable showing upload time in caption

# Initialize Telethon client
client = TelegramClient('uploader_session', api_id, api_hash)


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"COLAB Uploading {self.filename} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")

async def authenticate():
    try:
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=128 * 1024):  # Increased chunk size
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream['duration'])
            width = int(stream['width'])
            height = int(stream['height'])
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1

        # Use imageio to read the video frame at the given timestamp
        video_reader = imageio.get_reader(video_path)
        frame = video_reader.get_data(int(timestamp * video_reader.get_meta_data()['fps']))

        # Convert the frame to an image using Pillow
        image = Image.fromarray(frame)
        image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

        # Save the image to the temporary thumbnail file
        image.save(thumbnail_path)
        logger.info(f"Thumbnail generated at {thumbnail_path}")
        return thumbnail_path
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload COLAB {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=duration or 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create beautified caption
        if duration:
            video_duration = format_time_duration(duration)
            resolution = f"{width}x{height}" if width and height else "Unknown"

            caption = f"📁 **{movie_file_name}**\n\n"
            caption += f"📊 **Info:**\n"
            caption += f"• Size: {file_size_mb:.2f} MB\n"
            caption += f"• Duration: {video_duration}\n"
            caption += f"• Resolution: {resolution}\n"

            if SHOW_UPLOAD_TIME:
                caption += f"• Upload Time: {upload_time_formatted}\n"

            caption += f"\n🔄 Uploaded via COLAB"
        else:
            # Simplified caption if metadata extraction failed
            caption = f"📁 **{movie_file_name}**\n\n"
            caption += f"📊 **Info:**\n"
            caption += f"• Size: {file_size_mb:.2f} MB\n"

            if SHOW_UPLOAD_TIME:
                caption += f"• Upload Time: {upload_time_formatted}\n"

            caption += f"\n🔄 Uploaded via COLAB"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb,
                parse_mode='md'  # Enable Markdown for formatting
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                parse_mode='md'  # Enable Markdown for formatting
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        file_size = check_file_size(url)

        # Skip very large files
        if file_size and file_size > 200:  # Increased file size limit
            logger.warning(f"Skipping extremely large file: {url}")
            continue

        video_file = download_video(url)

        if video_file:
            await send_video(client, video_file, channel_id)

# Run the async function
if __name__ == '__main__':
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())

KeyboardInterrupt: 

In [ ]:
from pickle import FALSE
# @title 4. Upload url with thumbnail one line

import logging
import asyncio
import requests
import os
import subprocess
import json
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Telegram Bot and Telethon credentials
api_id = 1233728  # @param ["True","False"] {"type":"raw"} # Get from https://my.telegram.org
api_hash = '0959897ff7f45100d243daa78437fb7b' # @param ["True","False"] {"type":"raw"}  # Get from https://my.telegram.org
phone_number = "'+8801828767185'" # @param {"type":"string"}
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/url.txt'

# Option to enable/disable upload time in caption
# Set to False to disable showing upload time in caption
SHOW_UPLOAD_TIME = True # @param ["True","False"] {"type":"raw"}

# Initialize Telethon client
client = TelegramClient('uploader_session', api_id, api_hash)


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"COLAB Uploading {self.filename} [{current / (1024 * 1024):.2f}/{self.formatted_file_size}] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")

async def authenticate():
    try:
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=128 * 1024):  # Increased chunk size
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream['duration'])
            width = int(stream['width'])
            height = int(stream['height'])
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1

        # Use imageio to read the video frame at the given timestamp
        video_reader = imageio.get_reader(video_path)
        frame = video_reader.get_data(int(timestamp * video_reader.get_meta_data()['fps']))

        # Convert the frame to an image using Pillow
        image = Image.fromarray(frame)
        image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

        # Save the image to the temporary thumbnail file
        image.save(thumbnail_path)
        logger.info(f"Thumbnail generated at {thumbnail_path}")
        return thumbnail_path
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload COLAB {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=duration or 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time if enabled
        if SHOW_UPLOAD_TIME:
            caption += f" | Upload Time: {upload_time_formatted}"

        caption += " | COLAB"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        file_size = check_file_size(url)

        # Skip very large files
        if file_size and file_size > 200:  # Increased file size limit
            logger.warning(f"Skipping extremely large file: {url}")
            continue

        video_file = download_video(url)

        if video_file:
            await send_video(client, video_file, channel_id)

# Run the async function
if __name__ == '__main__':
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())

KeyboardInterrupt: 

In [ ]:
# @title 4. Upload url with thumbnail
import logging
import asyncio
import requests
import os
import subprocess
import json
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Telegram Bot and Telethon credentials
api_id = 1233728  # Get from https://my.telegram.org
api_hash = '0959897ff7f45100d243daa78437fb7b'  # Get from https://my.telegram.org
phone_number = '+8801828767185'  # Your phone number with country code
channel_id = -1002296651104  # Telegram channel ID

# Path to URL list file
url_list = '/content/url.txt'

# Initialize Telethon client
client = TelegramClient('uploader_session', api_id, api_hash)


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"COLAB Uploading {self.filename} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")

async def authenticate():
    try:
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_video_urls(filename=url_list):
    try:
        with open(filename, 'r') as file:
            urls = [line.strip() for line in file.readlines() if line.strip()]
        return urls
    except Exception as e:
        logger.error(f"Error reading {filename}: {e}")
        return []


def check_file_size(url):
    try:
        response = requests.head(url, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_video(url, download_folder='downloads'):
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_name = os.path.join(download_folder, url.split('/')[-1])
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(file_name, 'wb') as file:
                for chunk in response.iter_content(chunk_size=128 * 1024):  # Increased chunk size
                    file.write(chunk)
            return file_name
    except Exception as e:
        logger.error(f"Error downloading video from {url}: {e}")
    return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream['duration'])
            width = int(stream['width'])
            height = int(stream['height'])
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1

        # Use imageio to read the video frame at the given timestamp
        video_reader = imageio.get_reader(video_path)
        frame = video_reader.get_data(int(timestamp * video_reader.get_meta_data()['fps']))

        # Convert the frame to an image using Pillow
        image = Image.fromarray(frame)
        image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

        # Save the image to the temporary thumbnail file
        image.save(thumbnail_path)
        logger.info(f"Thumbnail generated at {thumbnail_path}")
        return thumbnail_path
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload COLAB {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=duration or 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create caption with file name and size
        caption = f"COLAB {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_videos():
    await authenticate()
    video_urls = await read_video_urls()

    for url in video_urls:
        file_size = check_file_size(url)

        # Skip very large files
        if file_size and file_size > 200:  # Increased file size limit
            logger.warning(f"Skipping extremely large file: {url}")
            continue

        video_file = download_video(url)

        if video_file:
            await send_video(client, video_file, channel_id)

# Run the async function
if __name__ == '__main__':
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_videos())


In [ ]:
# @title Upload URLs from CSV/List with Multi-Threaded Downloader

# @markdown ## Input Selection
input_type = "CSV File" # @param ["CSV File", "URL List"]

# @markdown ## File Path or URL List
file_path = "/content/direct_streamtape_links.csv" # @param {type:"string"}

# @markdown ## Upload Settings
include_sender_name = True # @param {type:"boolean"}
thread_count = 2 # @param {type:"slider", min:1, max:8, step:1}

# @markdown ## Account Selection
account_selection = "GP" # @param ["Robi", "GP"]

# @markdown ## Telegram channel ID
channel = "new" # @param ["old","new"]


import logging
import asyncio
import requests
import os
import subprocess
import json
import threading
import urllib.request
import urllib.parse
import re
import csv
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeVideo
from telethon.tl import types
from telethon import utils
import nest_asyncio
import imageio
from PIL import Image
import tempfile
import time
from datetime import timedelta

# Import the FastTelethon functions
from FastTelethon import upload_file

# Enable logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
                    level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Account configuration
ACCOUNTS = {
    "Robi": {
        "api_id": 1233728,
        "api_hash": "0959897ff7f45100d243daa78437fb7b",
        "phone_number": "+8801828767185",
        "session_name": "uploader_session_robi"
    },
    "GP": {
        "api_id": 21431565,
        "api_hash": "56d24b254423e0883a56a63046b38d7f",
        "phone_number": "+8801326573075",
        "session_name": "uploader_session_gp"
    }
}


CHANNELS= {
    "old": -1002296651104,
    "new": -1002259201126
}

# Use selected account
selected_account = ACCOUNTS[account_selection]
api_id = selected_account["api_id"]
api_hash = selected_account["api_hash"]
phone_number = selected_account["phone_number"]
session_name = selected_account["session_name"]

channel_id = CHANNELS[channel]

# Download settings
NUM_DOWNLOAD_THREADS = thread_count  # Number of threads to use for downloading
DOWNLOAD_FOLDER = 'downloads'  # Folder to store downloaded files

# Option to enable/disable sender name in caption
SHOW_SENDER_NAME = include_sender_name  # Set based on the checkbox value

# Initialize Telethon client with selected account
client = TelegramClient(session_name, api_id, api_hash)


def sanitize_filename(filename):
    """Remove invalid characters from a filename."""
    # Remove invalid characters
    invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']
    for char in invalid_chars:
        filename = filename.replace(char, '_')

    # Remove any leading/trailing spaces and dots
    filename = filename.strip('. ')

    # Ensure filename is not too long
    if len(filename) > 100:
        name, ext = os.path.splitext(filename)
        filename = name[:95] + ext

    return filename


class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=NUM_DOWNLOAD_THREADS):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        if self.file_size:
            self.chunks = self.get_chunks()
            self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True,
                                desc=f"Downloading {os.path.basename(filename)}")
        else:
            logger.error(f"Could not determine file size for {url}")

    def get_file_size(self):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            req = urllib.request.Request(self.url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req) as response:
                if 'Content-Length' in response.headers:
                    return int(response.headers['Content-Length'])
                return None
        except Exception as e:
            logger.error(f"Error getting file size: {e}")
            return None

    def get_chunks(self):
    # Use a more optimal chunk size (e.g., 8 MB chunks)
      optimal_chunk_size = 32 * 1024 * 1024  # 8 MB
      num_chunks = max(self.num_threads, self.file_size // optimal_chunk_size)
      chunk_size = self.file_size // num_chunks

      chunks = []
      for i in range(num_chunks):
          start = i * chunk_size
          # Important: Make sure there's no gap between chunks
          end = (i + 1) * chunk_size - 1 if i < num_chunks - 1 else self.file_size - 1
          chunks.append((start, end))
      return chunks

    def download_chunk(self, start, end, index):
        try:
            headers = {
                'Range': f"bytes={start}-{end}",
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            req = urllib.request.Request(self.url, headers=headers)
            with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    self.progress.update(len(chunk))
            logger.info(f"Chunk {index} downloaded")
        except Exception as e:
            logger.error(f"Error downloading chunk {index}: {e}")

    def merge_chunks(self):
        try:
            # Verify all part files exist before merging
            missing_parts = []
            for i in range(len(self.chunks)):
                part_filename = f"{self.filename}.part{i}"
                if not os.path.exists(part_filename):
                    missing_parts.append(i)

            if missing_parts:
                logger.error(f"Cannot merge: Missing part files: {missing_parts}")
                self.progress.close()
                return False

            # Check total size of parts matches expected file size
            total_parts_size = sum(os.path.getsize(f"{self.filename}.part{i}")
                                  for i in range(len(self.chunks)))

            if total_parts_size != self.file_size:
                logger.error(f"Size mismatch: Expected {self.file_size} bytes, got {total_parts_size} bytes")
                # Continue anyway, but log the error

            # Merge in correct order
            with open(self.filename, "wb") as final_file:
                for i in range(len(self.chunks)):
                    part_filename = f"{self.filename}.part{i}"
                    with open(part_filename, "rb") as part_file:
                        final_file.write(part_file.read())
                    # Remove part file after successful read
                    os.remove(part_filename)

            self.progress.close()
            logger.info(f"Download completed and merged: {self.filename}")
            return True
        except Exception as e:
            logger.error(f"Error merging chunks: {e}")
            self.progress.close()
            return False

    def verify_download(self):
        """Verify the downloaded file size matches the expected size"""
        if not os.path.exists(self.filename):
            return False

        actual_size = os.path.getsize(self.filename)
        if actual_size != self.file_size:
            logger.error(f"Verification failed: Expected {self.file_size} bytes, got {actual_size} bytes")
            return False
        return True

    def partial_download(self, start=None, end=None):
        """Download a specific part of the file to complete a partial download"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }

            # Add range header if specified
            if start is not None and end is not None:
                headers['Range'] = f"bytes={start}-{end}"
                logger.info(f"Attempting to download missing bytes {start}-{end}")

            req = urllib.request.Request(self.url, headers=headers)

            # Create a temporary file for this partial download
            temp_filename = f"{self.filename}.completion"
            with urllib.request.urlopen(req) as response, open(temp_filename, "wb") as f:
                total_size = int(response.headers.get('Content-Length', 0))
                progress = tqdm(total=total_size, unit='B', unit_scale=True,
                              desc=f"Completing download of {os.path.basename(self.filename)}")

                while True:
                    chunk = response.read(32768)
                    if not chunk:
                        break
                    f.write(chunk)
                    progress.update(len(chunk))

            progress.close()
            return temp_filename

        except Exception as e:
            logger.error(f"Error in partial download: {e}")
            return None

    def fix_download(self):
        """Attempt to fix a corrupted download by identifying and downloading missing parts"""
        try:
            if not os.path.exists(self.filename):
                logger.error("Cannot fix download: File doesn't exist")
                return False

            actual_size = os.path.getsize(self.filename)

            if actual_size > self.file_size:
                # File is larger than expected - truncate it
                logger.warning(f"File is larger than expected ({actual_size} > {self.file_size}). Truncating.")
                with open(self.filename, "r+b") as f:
                    f.truncate(self.file_size)
                return True

            elif actual_size < self.file_size:
                # File is smaller than expected - download the missing part
                logger.warning(f"File is smaller than expected ({actual_size} < {self.file_size}). Downloading missing part.")

                # Download the missing bytes
                temp_file = self.partial_download(start=actual_size, end=self.file_size-1)
                if not temp_file or not os.path.exists(temp_file):
                    return False

                # Append the missing bytes to the original file
                with open(self.filename, "ab") as original, open(temp_file, "rb") as completion:
                    original.write(completion.read())

                # Clean up
                os.remove(temp_file)

                # Verify again
                if os.path.getsize(self.filename) == self.file_size:
                    logger.info("File successfully completed")
                    return True

            return False

        except Exception as e:
            logger.error(f"Error fixing download: {e}")
            return False

    def start(self):
        if not self.file_size:
            return self.fallback_download()

        # Try multi-threaded download
        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        success = self.merge_chunks()

        # Verify and attempt to fix if needed
        if success and not self.verify_download():
            logger.warning("Download verification failed. Attempting to fix...")
            fix_success = self.fix_download()

            if fix_success and self.verify_download():
                logger.info("Download successfully fixed")
                return self.filename
            else:
                logger.warning("Could not fix download. Falling back to single-threaded download")
                if os.path.exists(self.filename):
                    os.remove(self.filename)
                return self.fallback_download()

        return self.filename if success else self.fallback_download()

    def fallback_download(self):
        """Fallback to regular download if multi-threaded download fails"""
        logger.info(f"Using fallback download method for {self.url}")
        try:
            with tqdm(unit='B', unit_scale=True, desc=f"Downloading {os.path.basename(self.filename)}") as progress:
                def report_hook(count, block_size, total_size):
                    if total_size > 0:
                        progress.total = total_size
                        progress.update(block_size)

                headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', headers['User-Agent'])]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.url, self.filename, reporthook=report_hook)
            return self.filename
        except Exception as e:
            logger.error(f"Fallback download failed: {e}")
            return None


class UploadProgress:
    def __init__(self, message, filename):
        self.message = message
        self.filename = filename
        self.last_update = 0
        self.start_time = asyncio.get_event_loop().time()
        self.file_size = os.path.getsize(filename)
        self.formatted_file_size = f"{self.file_size / (1024 * 1024):.2f} MB"
        self.upload_complete = False
        self.upload_duration = 0

    async def progress_callback(self, current, total):
        current_time = asyncio.get_event_loop().time()

        # Update progress every 3 seconds to avoid rate limits
        if current_time - self.last_update > 3:
            try:
                percentage = int(current * 100 / total)
                elapsed_time = current_time - self.start_time
                speed = (current / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0

                await self.message.edit(f"{account_selection} Uploading {os.path.basename(self.filename)} [{current / (1024 * 1024):.2f}/{self.formatted_file_size} MB] : {percentage}% | Speed: {speed:.2f} MB/s")
                self.last_update = current_time

                # Check if upload is complete
                if current >= total:
                    self.upload_complete = True
                    self.upload_duration = elapsed_time
            except Exception as e:
                logger.error(f"Error updating progress: {e}")


async def authenticate():
    try:
        logger.info(f"Authenticating with {account_selection} account ({phone_number})")
        await client.start(phone=phone_number)
    except Exception as e:
        logger.error(f"Authentication error: {e}")
        raise


async def read_csv_file(filename=file_path):
    """Read the CSV file and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
            csv_reader = csv.reader(csvfile)
            for row in csv_reader:
                if len(row) >= 2:
                    # First column is filename, second column is URL
                    filename = sanitize_filename(row[0])
                    url = row[1].strip()
                    entries.append((filename, url))

        logger.info(f"Read {len(entries)} entries from CSV file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading CSV file {filename}: {e}")
        return []


async def read_url_list(filename=file_path):
    """Read a text file with URLs and return a list of (filename, url) tuples."""
    try:
        entries = []
        with open(filename, 'r', encoding='utf-8') as file:
            for line in file:
                url = line.strip()
                if url and not url.startswith('#'):
                    # Extract filename from URL
                    parsed_url = urllib.parse.urlparse(url)
                    path = parsed_url.path
                    filename = os.path.basename(path)

                    # If no filename could be extracted, use the domain with timestamp
                    if not filename or filename == '':
                        domain = parsed_url.netloc.split('.')[-2] if len(parsed_url.netloc.split('.')) > 1 else 'file'
                        timestamp = int(time.time())
                        filename = f"{domain}_{timestamp}.mp4"

                    filename = sanitize_filename(filename)
                    entries.append((filename, url))

        logger.info(f"Read {len(entries)} entries from URL list file: {filename}")
        return entries
    except Exception as e:
        logger.error(f"Error reading URL list file {filename}: {e}")
        return []


def check_file_size(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.head(url, headers=headers, allow_redirects=True)
        if 'Content-Length' in response.headers:
            return int(response.headers['Content-Length']) / (1024 * 1024)
        return None
    except Exception as e:
        logger.error(f"Error checking file size for {url}: {e}")
        return None


def download_file(filename, url, download_folder=DOWNLOAD_FOLDER):
    """Download file using multi-threaded downloader with the specified filename"""
    try:
        os.makedirs(download_folder, exist_ok=True)
        file_path = os.path.join(download_folder, filename)

        # Make sure we don't overwrite existing files
        if os.path.exists(file_path):
            base, ext = os.path.splitext(filename)
            timestamp = int(time.time())
            filename = f"{base}_{timestamp}{ext}"
            file_path = os.path.join(download_folder, filename)

        # Use multi-threaded downloader
        downloader = MultiThreadedDownloader(url, file_path, NUM_DOWNLOAD_THREADS)
        result = downloader.start()

        return result
    except Exception as e:
        logger.error(f"Error downloading file from {url}: {e}")
        return None


def get_video_metadata(file_path):
    try:
        command = [
            'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
            'stream=duration,width,height', '-of', 'json', file_path
        ]
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata = json.loads(result.stdout)
        if 'streams' in metadata and len(metadata['streams']) > 0:
            stream = metadata['streams'][0]
            duration = float(stream.get('duration', 0))
            width = int(stream.get('width', 0))
            height = int(stream.get('height', 0))
            return duration, width, height
    except Exception as e:
        logger.error(f"Error extracting metadata for {file_path}: {e}")
    return None, None, None


def format_time_duration(seconds):
    """Format seconds into a human-readable time format."""
    delta = timedelta(seconds=int(seconds))
    hours, remainder = divmod(delta.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}h {minutes}m {seconds}s"
    elif minutes > 0:
        return f"{minutes}m {seconds}s"
    else:
        return f"{seconds}s"


def generate_thumbnail(video_path, duration):
    """
    Generate a thumbnail from the video at 10% of its duration.
    Uses a temporary file to avoid storing thumbnails permanently.
    """
    try:
        # Check if video file exists
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        # Create a temporary file for the thumbnail
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_thumb:
            thumbnail_path = temp_thumb.name
            logger.info(f"Generated temp thumbnail file: {thumbnail_path}")

        # Calculate timestamp for thumbnail (10% of video duration)
        timestamp = duration * 0.1 if duration else 5  # Default to 5 seconds if duration unknown

        # Use ffmpeg to extract a thumbnail
        try:
            command = [
                'ffmpeg', '-y', '-ss', str(timestamp), '-i', video_path,
                '-vframes', '1', '-vf', 'scale=320:-1', thumbnail_path
            ]
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            logger.info(f"Thumbnail generated at {thumbnail_path} using ffmpeg")
            return thumbnail_path
        except subprocess.CalledProcessError:
            logger.warning("ffmpeg failed, trying imageio as fallback")

            # Fallback to imageio if ffmpeg fails
            try:
                video_reader = imageio.get_reader(video_path)
                fps = video_reader.get_meta_data().get('fps', 30)
                frame_number = int(timestamp * fps) if fps else 0

                if frame_number >= len(video_reader):
                    frame_number = 0

                frame = video_reader.get_data(frame_number)

                # Convert the frame to an image using Pillow
                image = Image.fromarray(frame)
                image = image.resize((320, int(320 * frame.shape[0] / frame.shape[1])))  # Resize to width=320 and keep aspect ratio

                # Save the image to the temporary thumbnail file
                image.save(thumbnail_path)
                logger.info(f"Thumbnail generated at {thumbnail_path} using imageio")
                return thumbnail_path
            except Exception as e:
                logger.error(f"Imageio thumbnail generation failed: {e}")
                return None
    except Exception as e:
        logger.error(f"Error generating thumbnail: {e}")
        return None


async def send_video(client, file_path, channel_id):
    try:
        movie_file_name = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        duration, width, height = get_video_metadata(file_path)

        # Prepare progress tracking
        progress_message = await client.send_message(channel_id, f"Preparing to upload {account_selection} {movie_file_name}...")
        upload_progress = UploadProgress(progress_message, file_path)

        # Track upload start time
        upload_start_time = asyncio.get_event_loop().time()

        # Generate thumbnail
        thumbnail_path = generate_thumbnail(file_path, duration) if duration else None

        # Open file for reading
        with open(file_path, 'rb') as file:
            # Use FastTelethon's upload_file with progress callback
            uploaded_file = await upload_file(
                client,
                file,
                progress_callback=upload_progress.progress_callback
            )

        # Calculate upload duration
        upload_end_time = asyncio.get_event_loop().time()
        upload_duration = upload_end_time - upload_start_time
        upload_time_formatted = format_time_duration(upload_duration)

        # Get file attributes
        attributes, mime_type = utils.get_attributes(file_path)

        # Define video attributes
        video_attributes = DocumentAttributeVideo(
            duration=int(duration) if duration else 0,
            w=width or 0,
            h=height or 0,
            supports_streaming=True
        )

        # Combine attributes
        final_attributes = attributes + [video_attributes]

        # Create simplified caption in one line
        caption = f"📁 {movie_file_name} | Size: {file_size_mb:.2f} MB"

        # Add upload time
        caption += f" | Upload Time: {upload_time_formatted}"

        # Add sender name if enabled
        if SHOW_SENDER_NAME:
            caption += f" | {account_selection}"

        # Prepare media for sending
        media = types.InputMediaUploadedDocument(
            file=uploaded_file,
            mime_type=mime_type,
            attributes=final_attributes,
            force_file=False
        )

        # Upload thumbnail if generated
        if thumbnail_path and os.path.exists(thumbnail_path):
            with open(thumbnail_path, 'rb') as thumb_file:
                thumb = await upload_file(client, thumb_file)

            # Send the file with thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption,
                thumb=thumb
            )

            # Remove temporary thumbnail
            os.unlink(thumbnail_path)
        else:
            # Send the file without thumbnail
            await client.send_file(
                entity=channel_id,
                file=media,
                caption=caption
            )

        # Delete progress message and local file
        await progress_message.delete()
        os.remove(file_path)

    except Exception as e:
        logger.error(f"Failed to send video: {e}")
        # Optionally, send an error message to the channel
        await client.send_message(channel_id, f"Upload failed for {movie_file_name}: {str(e)}")


async def process_and_send_files():
    await authenticate()

    # Process files based on input type
    if input_type == "CSV File":
        file_entries = await read_csv_file()
    else:  # URL List
        file_entries = await read_url_list()

    for filename, url in file_entries:
        try:
            # Check file size - use HEAD request method
            file_size = check_file_size(url)

            # Skip very large files
            if file_size and file_size > 2000:  # Maximum file size limit of 2000MB (2GB)
                logger.warning(f"Skipping extremely large file: {filename} ({file_size:.2f} MB)")
                continue

            logger.info(f"Processing URL: {url} with filename: {filename}")

            # Download the file using the multi-threaded downloader
            logger.info(f"Starting download of {filename} from {url}")
            downloaded_file = download_file(filename, url)

            if downloaded_file:
                logger.info(f"Successfully downloaded {url} to {downloaded_file}, uploading to Telegram")
                await send_video(client, downloaded_file, channel_id)
            else:
                logger.error(f"Failed to download {url}")
        except Exception as e:
            logger.error(f"Error processing {filename} from {url}: {e}")

# Run the async function
if __name__ == '__main__':
    logger.info(f"Starting upload process with {account_selection} account")
    logger.info(f"Using {NUM_DOWNLOAD_THREADS} download threads")
    logger.info(f"Input type: {input_type}")
    logger.info(f"File path: {file_path}")
    logger.info(f"Show sender name: {SHOW_SENDER_NAME}")

    # Create downloads directory if it doesn't exist
    os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
    nest_asyncio.apply()  # Apply nest_asyncio patch
    loop = asyncio.get_event_loop()
    loop.run_until_complete(process_and_send_files())

ERROR:__main__:Error checking file size for Direct Link: Invalid URL 'Direct Link': No scheme supplied. Perhaps you meant https://Direct Link?
ERROR:__main__:Error getting file size: unknown url type: 'Direct Link'
ERROR:__main__:Could not determine file size for Direct Link
ERROR:__main__:Fallback download failed: unknown url type: 'Direct Link'
ERROR:__main__:Failed to download Direct Link

KeyboardInterrupt: 

# Utils

In [ ]:
import time
from IPython.display import display, clear_output

# Function to read current network usage
def get_network_usage():
    try:
        with open('/proc/net/dev', 'r') as f:
            lines = f.readlines()

        for line in lines:
            if 'eth0' in line:  # 'eth0' is the main Colab network interface
                data = line.split()
                download = int(data[1])  # Bytes received
                upload = int(data[9])  # Bytes sent
                return download, upload

        return 0, 0  # Default if no data found
    except Exception as e:
        print(f"ERROR: {e}")
        return 0, 0

# Start monitoring (updates every second)
prev_download, prev_upload = get_network_usage()

while True:
    time.sleep(1)  # Measure per second
    curr_download, curr_upload = get_network_usage()

    download_speed = (curr_download - prev_download) / 1024 / 1024  # Convert to MB/s
    upload_speed = (curr_upload - prev_upload) / 1024 / 1024  # Convert to MB/s

    prev_download, prev_upload = curr_download, curr_upload  # Update previous values

    # Display results dynamically
    clear_output(wait=True)
    print("📡 Real-time Internet Speed (Colab)")
    print(f"⬇️ Download Speed: {download_speed:.3f} MB/s")
    print(f"⬆️ Upload Speed: {upload_speed:.3f} MB/s")

    # Stop after a few iterations to prevent infinite looping
    # Remove this if you want it to run indefinitely
    if download_speed == 0 and upload_speed == 0:
        print("⚠️ No network activity detected. Stopping...")
        break


🔄 **Real-time Internet Speed in Google Colab** 🔄
⬇️ Download Speed: 0.001 MB/s
⬆️ Upload Speed: 0.020 MB/s


KeyboardInterrupt: 

In [ ]:
import time
import threading
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets
from IPython import get_ipython

# Function to read current network usage
def get_network_usage():
    try:
        with open('/proc/net/dev', 'r') as f:
            lines = f.readlines()

        for line in lines:
            if 'eth0' in line:  # 'eth0' is the main Colab network interface
                data = line.split()
                download = int(data[1])  # Bytes received
                upload = int(data[9])  # Bytes sent
                return download, upload

        return 0, 0  # Default if no data found
    except Exception as e:
        print(f"ERROR: {e}")
        return 0, 0

# Create output widgets
output_area = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '10px', 'background': '#f8f8f8'})
status_label = widgets.HTML(value="<h3>📡 Real-time Network Monitor</h3>")
download_label = widgets.HTML(value="⬇️ Download: 0.000 MB/s")
upload_label = widgets.HTML(value="⬆️ Upload: 0.000 MB/s")
stop_button = widgets.Button(description="Stop Monitoring", button_style="danger")

# Global control variable
monitoring_active = True

# Function to update the display
def update_display(download_speed, upload_speed):
    download_label.value = f"⬇️ Download: {download_speed:.3f} MB/s"
    upload_label.value = f"⬆️ Upload: {upload_speed:.3f} MB/s"

# Monitoring function that runs in the background
def monitor_network():
    global monitoring_active
    prev_download, prev_upload = get_network_usage()

    while monitoring_active:
        time.sleep(1)
        curr_download, curr_upload = get_network_usage()

        download_speed = (curr_download - prev_download) / 1024 / 1024  # Convert to MB/s
        upload_speed = (curr_upload - prev_upload) / 1024 / 1024  # Convert to MB/s

        prev_download, prev_upload = curr_download, curr_upload

        # Use IPython's execute method to update UI from the background thread
        get_ipython().run_cell_magic(
            'javascript',
            '',
            f'''
            let downloadLabel = document.querySelector('[data-widget-id="{download_label._model_id}"] div');
            let uploadLabel = document.querySelector('[data-widget-id="{upload_label._model_id}"] div');
            if (downloadLabel) downloadLabel.innerHTML = "⬇️ Download: {download_speed:.3f} MB/s";
            if (uploadLabel) uploadLabel.innerHTML = "⬆️ Upload: {upload_speed:.3f} MB/s";
            '''
        )

# Stop button callback
def on_stop_button_clicked(b):
    global monitoring_active
    monitoring_active = False
    status_label.value = "<h3>📡 Network Monitor (Stopped)</h3>"

stop_button.on_click(on_stop_button_clicked)

# Start monitoring in a separate thread
def start_monitoring():
    global monitoring_active
    monitoring_active = True

    # Create and display the dashboard
    dashboard = widgets.VBox([status_label, download_label, upload_label, stop_button])
    display(dashboard)

    # Start the monitoring thread
    thread = threading.Thread(target=monitor_network)
    thread.daemon = True  # Thread will exit when main program exits
    thread.start()

    return "Network monitor started in background. You can continue using other cells."

# Call this function to start monitoring
start_monitoring()

'Network monitor started in background. You can continue using other cells.'

In [ ]:
import os
import threading
import urllib.request
from tqdm import tqdm

class MultiThreadedDownloader:
    def __init__(self, url, filename, num_threads=4):
        self.url = url
        self.filename = filename
        self.num_threads = num_threads
        self.file_size = self.get_file_size()
        self.chunks = self.get_chunks()
        self.progress = tqdm(total=self.file_size, unit='B', unit_scale=True, desc="Downloading")

    def get_file_size(self):
        req = urllib.request.Request(self.url, method='HEAD')
        with urllib.request.urlopen(req) as response:
            return int(response.headers['Content-Length'])

    def get_chunks(self):
        chunk_size = self.file_size // self.num_threads
        chunks = []
        for i in range(self.num_threads):
            start = i * chunk_size
            end = (start + chunk_size - 1) if i < self.num_threads - 1 else self.file_size - 1
            chunks.append((start, end))
        return chunks

    def download_chunk(self, start, end, index):
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={start}-{end}"})
        with urllib.request.urlopen(req) as response, open(f"{self.filename}.part{index}", "wb") as f:
            while True:
                chunk = response.read(1024)
                if not chunk:
                    break
                f.write(chunk)
                self.progress.update(len(chunk))
        print(f"Chunk {index} downloaded")

    def merge_chunks(self):
        with open(self.filename, "wb") as final_file:
            for i in range(self.num_threads):
                part_filename = f"{self.filename}.part{i}"
                with open(part_filename, "rb") as part_file:
                    final_file.write(part_file.read())
                os.remove(part_filename)
        self.progress.close()
        print("Download completed and merged!")

    def start(self):
        threads = []
        for i, (start, end) in enumerate(self.chunks):
            thread = threading.Thread(target=self.download_chunk, args=(start, end, i))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        self.merge_chunks()

if __name__ == "__main__":
    url = "https://streamtape.to/get_video?id=j2yGg7WLB3im0j&expires=1741415999&ip=FHEsD0OAKxSHDN&token=hZnIKTadM2zZ&token=hZnIKTadM2Vg&dl=1sp"  # Replace with actual file URL
    filename = "file.zip"
    downloader = MultiThreadedDownloader(url, filename, num_threads=8)
    downloader.start()


Downloading: 100%|█████████▉| 427M/429M [01:27<00:00, 3.20MB/s]

Chunk 4 downloaded
Chunk 0 downloaded


Downloading: 100%|█████████▉| 428M/429M [01:27<00:00, 3.50MB/s]

Chunk 7 downloaded
Chunk 6 downloaded
Chunk 2 downloaded
Chunk 3 downloaded
Chunk 1 downloaded
Chunk 5 downloaded


Downloading: 100%|██████████| 429M/429M [01:30<00:00, 4.73MB/s]

Download completed and merged!


In [ ]:
import logging
from telethon import TelegramClient
from telethon.tl.types import InputMediaUploadedPhoto, InputMediaUploadedDocument
from telethon.tl.types import DocumentAttributeVideo

# Configure logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Replace these with your own values
api_id = 1233728
api_hash = '0959897ff7f45100d243daa78437fb7b'
phone_number = '+8801828767185'

# Create the client and connect
client = TelegramClient('uploader_session_robi', api_id, api_hash)

async def main():
    try:
        logger.error("Starting the client...")
        await client.start(phone_number)
        logger.error("Client started successfully.")

        # Replace these with your file paths
        image_paths  = [
            '/content/thumbs/image_12.png', '/content/thumbs/image_13.png', '/content/thumbs/image_14.png',
            '/content/thumbs/image_15.png', '/content/thumbs/image_16.png' ]
        video_path = '/content/downloads/49016.mp4'

        media_group = []

        # Upload images and add them to the media group
        for image_path in image_paths:
            try:
                logger.debug(f"Uploading image: {image_path}")
                uploaded_file = await client.upload_file(image_path)
                media_group.append(InputMediaUploadedPhoto(uploaded_file))
                logger.debug(f"Image {image_path} uploaded successfully.")
            except Exception as e:
                logger.error(f"Failed to upload image {image_path}: {e}")

        # Upload the video and add it to the media group
        try:
            logger.debug(f"Uploading video: {video_path}")
            uploaded_video = await client.upload_file(video_path)
            video_attributes = [
                DocumentAttributeVideo(duration=10, w=1280, h=720)  # Replace with actual video attributes
            ]
            media_group.append(InputMediaUploadedDocument(
                file=uploaded_video,
                mime_type='video/mp4',  # Replace with the correct MIME type
                attributes=video_attributes
            ))
            logger.debug(f"Video {video_path} uploaded successfully.")
        except Exception as e:
            logger.error(f"Failed to upload video {video_path}: {e}")

        # Replace 'TARGET_CHANNEL' with the channel's username or ID
        target_channel = -1002259201126  # Example: Use '@my_channel' or '-1001234567890'

        # Send the media group with a caption to the target channel
        try:
            logger.debug(f"Sending media group to channel {target_channel}...")
            await client.send_file(
                target_channel,  # Send to the specified channel
                media_group,
                caption="This is the caption for the media group!"
            )
            logger.error(f"Media group sent successfully to {target_channel}!")
        except Exception as e:
            logger.error(f"Failed to send media group to channel {target_channel}: {e}")

    except Exception as e:
        logger.error(f"An error occurred in the main function: {e}")

# Run the client
async def run():
    try:
        logger.error("Running the client...")
        async with client:
            await main()
    except Exception as e:
        logger.error(f"An error occurred while running the client: {e}")

# Start the event loop
import asyncio
try:
    logger.error("Starting the event loop...")
    await run()
except Exception as e:
    logger.error(f"An error occurred in the event loop: {e}")

ERROR:__main__:Starting the event loop...
ERROR:__main__:Running the client...
ERROR:__main__:Starting the client...
ERROR:__main__:Client started successfully.
